
# ATfinance — One-Click Full Run Notebook

Run the engine cells first, then the final launcher cell.  
This notebook:
- builds the engine outputs
- writes the Streamlit app
- launches the UI from the same notebook
- keeps all 7 pages
- keeps explainability first
- uses the shared `/tmp/agentic_sector` output folder


# ATfinance — Adaptive Financial Intelligence Engine
### Created by Aayush Tiwari · v5 Fixed Edition

**Architecture:** Multi-Source News → FinBERT Semantic Engine → Adaptive Quant → Forecast Tournament → Master Fusion

**Run Order:** Block 0 → 1-4 → 5 → 6 → 7 → 7.1 → 8 → 8.5 → UI

> All yfinance downloads are cached in memory — universe data is downloaded **once** total, not per block.

In [ ]:
# ============================================================
# BLOCK 0: ONE-TIME SETUP (run once per Colab session)
# Version-checked install so Colab doesn't get pinned to pandas 3.x.
# ============================================================
import sys, subprocess, importlib, os

REQUIRED = {
    "pandas": "2.2.2",
    "numpy": "1.26.4",
    "scikit-learn": "1.5.1",
    "scipy": "1.13.1",
    "statsmodels": "0.14.2",
}

OPTIONAL_IMPORTS = [
    ("feedparser", "feedparser"),
    ("requests", "requests"),
    ("lxml", "lxml"),
    ("sentence_transformers", "sentence-transformers"),
    ("transformers", "transformers"),
    ("torch", "torch"),
    ("yfinance", "yfinance"),
    ("pandas_ta", "pandas-ta"),
    ("xgboost", "xgboost"),
    ("plotly", "plotly"),
    ("ipywidgets", "ipywidgets"),
    ("streamlit", "streamlit"),
    ("trafilatura", "trafilatura"),
    ("newspaper", "newspaper3k"),
    ("bs4", "beautifulsoup4"),
]

def _ver(v):
    import re
    return tuple(int(x) for x in re.findall(r"\d+", str(v))[:3])

need = []
for mod, pinned in REQUIRED.items():
    try:
        m = importlib.import_module(mod if mod != "scikit-learn" else "sklearn")
        cur = getattr(m, "__version__", "0")
        if _ver(cur) != _ver(pinned):
            need.append(f"{mod}=={pinned}")
    except Exception:
        need.append(f"{mod}=={pinned}")

for imp_name, pip_name in OPTIONAL_IMPORTS:
    try:
        importlib.import_module(imp_name)
    except Exception:
        need.append(pip_name)

if need:
    print("📦 Installing / pinning required packages...")
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall"] + need
    subprocess.run(cmd, check=True)
    print("✅ Packages pinned. Please restart runtime once, then rerun from Block 0.")
else:
    print("✅ Environment already matches the pinned ATfinance stack.")

os.makedirs("/tmp/agentic_sector", exist_ok=True)
os.makedirs("/tmp/agentic_sector/reports", exist_ok=True)
print("✅ All dependencies ready. Proceed to Block 1-4.")

📦 Installing feedparser (verbose)...
📦 Installing remaining dependencies (one-time)...
  Using cached pandas_ta-0.4.71b0-py3-none-any.whl.metadata (2.3 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 9.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: llvmlite
    Found existing installation: 

In [ ]:
# ============================================================
# BLOCK 1-4: MULTI-SOURCE NEWS INGESTOR
# Sources: Google News RSS + Moneycontrol RSS + Economic Times RSS
#          + NSE announcements feed
# Robust: each source is independent; one failure doesn't stop run
# ============================================================
# deps installed in Block 0 — no re-install needed

import feedparser, requests, re, os, time, hashlib, base64
import pandas as pd
from datetime import datetime, timezone, timedelta
import numpy as np
import json

OUT_DIR = "/tmp/agentic_sector"
os.makedirs(OUT_DIR, exist_ok=True)

NIFTY_UNIVERSE = {
    "ADANIENT.NS":"Adani Enterprises","ADANIPORTS.NS":"Adani Ports",
    "APOLLOHOSP.NS":"Apollo Hospitals","ASIANPAINT.NS":"Asian Paints",
    "AXISBANK.NS":"Axis Bank","BAJAJ-AUTO.NS":"Bajaj Auto",
    "BAJFINANCE.NS":"Bajaj Finance","BAJAJFINSV.NS":"Bajaj Finserv",
    "BEL.NS":"Bharat Electronics","BHARTIARTL.NS":"Bharti Airtel",
    "BPCL.NS":"Bharat Petroleum","BRITANNIA.NS":"Britannia",
    "CIPLA.NS":"Cipla","COALINDIA.NS":"Coal India","DRREDDY.NS":"Dr Reddy",
    "EICHERMOT.NS":"Eicher Motors","GRASIM.NS":"Grasim Industries",
    "HCLTECH.NS":"HCL Technologies","HDFCBANK.NS":"HDFC Bank",
    "HDFCLIFE.NS":"HDFC Life Insurance","HEROMOTOCO.NS":"Hero MotoCorp",
    "HINDALCO.NS":"Hindalco Industries","HINDUNILVR.NS":"Hindustan Unilever",
    "ICICIBANK.NS":"ICICI Bank","INDUSINDBK.NS":"IndusInd Bank",
    "INFY.NS":"Infosys","ITC.NS":"ITC Limited","JSWSTEEL.NS":"JSW Steel",
    "KOTAKBANK.NS":"Kotak Mahindra Bank","LT.NS":"Larsen and Toubro",
    "M&M.NS":"Mahindra Mahindra","MARUTI.NS":"Maruti Suzuki",
    "NESTLEIND.NS":"Nestle India","NTPC.NS":"NTPC Limited",
    "ONGC.NS":"Oil Natural Gas Corporation","POWERGRID.NS":"Power Grid",
    "RELIANCE.NS":"Reliance Industries","SBILIFE.NS":"SBI Life Insurance",
    "SBIN.NS":"State Bank India","SHRIRAMFIN.NS":"Shriram Finance",
    "SUNPHARMA.NS":"Sun Pharmaceutical","TATACONSUM.NS":"Tata Consumer",
    "TMPV.NS":"Tata Motors Passenger Vehicles",
    "TMCV.NS":"Tata Motors Commercial Vehicles","TATASTEEL.NS":"Tata Steel",
    "TCS.NS":"Tata Consultancy Services","TECHM.NS":"Tech Mahindra",
    "TITAN.NS":"Titan Company","TRENT.NS":"Trent Limited",
    "ULTRACEMCO.NS":"UltraTech Cement","WIPRO.NS":"Wipro"
}

TRUSTED = ['economic times','bloomberg','moneycontrol','livemint','mint',
           'cnbc','reuters','business standard','financial express',
           'ndtv','businessline','the hindu','yahoo finance','bq prime',
           'et markets','nse','bse','times of india','hindu business line']
BANNED  = ['market mojo','simply wall st','trendlyne','zacks',
           'seeking alpha','stocktwits','motilal oswal','smallcase']


MAX_PER_TICKER = 20
MAX_TOTAL_ARTICLES = 1000

ARTICLE_PRIORITY_TERMS = {
    'earnings': 2.0, 'result': 1.8, 'results': 1.8, 'revenue': 1.8, 'profit': 1.8,
    'q4': 2.2, 'q3': 2.0, 'q2': 1.9, 'q1': 1.9, 'guidance': 2.0, 'upgrade': 2.0,
    'downgrade': 1.8, 'target': 1.8, 'order': 1.6, 'deal': 1.5, 'contract': 1.5,
    'buyback': 1.9, 'dividend': 1.8, 'bonus': 1.6, 'split': 1.6, 'stake': 1.3,
    'approval': 1.4, 'launch': 1.3, 'acquisition': 1.7, 'merger': 1.7, 'funding': 1.5,
    'margin': 1.5, 'sales': 1.4, 'sebi': 1.3, 'rbi': 1.3, 'investigation': 1.2,
    'profitability': 1.5, 'loss': 1.2, 'expansion': 1.3, 'partnership': 1.4,
}

SOURCE_BONUS = {
    'REUTERS': 2.4,
    'BLOOMBERG': 2.2,
    'CNBC': 1.8,
    'CNBC-TV18': 1.8,
    'MONEYCONTROL': 1.7,
    'ECONOMIC TIMES': 1.5,
    'ET MARKETS': 1.5,
    'BUSINESS STANDARD': 1.4,
    'FINANCIAL EXPRESS': 1.3,
    'BQ PRIME': 1.3,
    'BUSINESSLINE': 1.2,
    'THE HINDU': 1.1,
    'YAHOO FINANCE': 1.0,
    'NSE': 1.0,
    'BSE': 0.9,
}

STOPWORDS = {
    'the','and','for','with','from','into','that','this','will','over','after',
    'stock','stocks','share','shares','price','company','companies','market',
    'indian','india','nse','bse','today','update','updates','news','may','also',
    'its','their','his','her','have','has','had','was','were','been','but','not',
    'about','what','why','how','can','could','would','should','into','more','new'
}

# clean_html defined above normalize_text (order fix)
def clean_html(text):
    text = re.sub(r'<[^>]+>', ' ', str(text))
    return re.sub(r'\s+', ' ', text).strip()


def normalize_text(s):
    s = clean_html(s)
    s = re.sub(r'[^a-zA-Z0-9 ]+', ' ', s.lower())
    s = re.sub(r'\s+', ' ', s).strip()
    return s



def decode_google_news_url(raw_link):
    """Decode Google News RSS URLs from query param or base64 path blob."""
    raw_link = str(raw_link or "").strip()
    if not raw_link:
        return raw_link
    try:
        from urllib.parse import urlparse, parse_qs, unquote
        qs = parse_qs(urlparse(raw_link).query)
        if "url" in qs and qs["url"]:
            candidate = unquote(qs["url"][0])
            if candidate.startswith("http"):
                return candidate
    except Exception:
        pass
    try:
        m = re.search(r"/articles/([^/?#]+)", raw_link)
        if m:
            blob = m.group(1)
            pad = "=" * (-len(blob) % 4)
            decoded = base64.urlsafe_b64decode((blob + pad).encode("utf-8")).decode("utf-8", errors="ignore")
            url_m = re.search(r"https?://[^\s\'\"<>]+", decoded)
            if url_m:
                return url_m.group(0).rstrip(").,")
    except Exception:
        pass
    return raw_link

def title_signature(title):
    tokens = [t for t in normalize_text(title).split() if len(t) > 2 and t not in STOPWORDS]
    if not tokens:
        return ""
    return " ".join(tokens[:12])

def company_tokens(company):
    toks = [t for t in normalize_text(company).split() if len(t) > 2 and t not in STOPWORDS]
    # preserve order but reduce repetitions
    seen = set()
    out = []
    for t in toks:
        if t not in seen:
            out.append(t)
            seen.add(t)
    return out

def recency_bonus(published_at):
    dt = pd.to_datetime(published_at, errors='coerce', utc=True)
    if pd.isna(dt):
        return 0.6
    now_utc = pd.Timestamp.now(tz='UTC')
    age_hours = max((now_utc - dt).total_seconds() / 3600.0, 0.0)
    # 48h freshness boost, slowly decaying thereafter
    return float(np.exp(-age_hours / 48.0) * 2.0 + 0.25)

def source_bonus(source):
    src = str(source).upper().strip()
    for key, bonus in SOURCE_BONUS.items():
        if key in src:
            return bonus
    return 0.5

def keyword_bonus(text):
    txt = normalize_text(text)
    score = 0.0
    for term, weight in ARTICLE_PRIORITY_TERMS.items():
        if term in txt:
            score += weight
    # avoid runaway scores
    return min(score, 8.0)

def ticker_bonus(title, body, company, ticker):
    txt = normalize_text(f"{title} {body}")
    bonus = 0.0
    ticker_base = str(ticker).replace('.NS', '').lower()
    if ticker_base and ticker_base in txt:
        bonus += 2.0
    for tok in company_tokens(company):
        if tok in txt:
            bonus += 0.8
    # title carries more weight than body
    if normalize_text(title):
        for tok in company_tokens(company):
            if tok in normalize_text(title):
                bonus += 0.8
    return min(bonus, 5.0)

def article_length_bonus(text):
    wc = len(str(text).split())
    if wc < 20:
        return -1.5
    if wc < 40:
        return -0.5
    if wc <= 180:
        return 1.0
    return 0.7

def compute_relevance_score(row):
    title = row.get('title', '')
    body = row.get('raw_text', row.get('nlp_text', ''))
    company = row.get('target_company', '')
    ticker = row.get('ticker', '')
    source = row.get('source', '')
    score = 0.0
    score += source_bonus(source)
    score += recency_bonus(row.get('published_at', None))
    score += keyword_bonus(f"{title} {body}")
    score += ticker_bonus(title, body, company, ticker)
    score += article_length_bonus(body)
    # light boost for clearer headline-centric items
    if len(str(title).split()) >= 4:
        score += 0.5
    return float(score)


def title_hash(t):
    return hashlib.md5(re.sub(r'\W','',t.lower()).encode()).hexdigest()

def fetch_google_news(company, ticker, seen):
    records = []
    strategies = [
        f'"{company}" stock NSE',
        f'"{company}" earnings revenue profit',
        f'"{company}" share price',
    ]
    import urllib.parse
    for q in strategies:
        if len(records) >= 12: break
        url = f"https://news.google.com/rss/search?q={urllib.parse.quote(q)}&hl=en-IN&gl=IN&ceid=IN:en"
        try:
            feed = feedparser.parse(url)
            for e in feed.entries:
                title = re.sub(r'\s*-\s*[^-]+$','', e.get('title',''))
                if title_hash(title) in seen: continue
                pub = str(e.source.get('title','') if hasattr(e,'source') else '').lower()
                # fallback: extract publisher from title suffix
                if not pub:
                    m = re.search(r'-\s*(.+)$', e.get('title',''))
                    pub = m.group(1).lower() if m else ''
                if any(b in pub for b in BANNED): continue
                if not any(t in pub for t in TRUSTED): continue
                raw = clean_html(e.get('summary',''))
                # 120-word window (was 40 — now captures EPS, targets, revenue)
                full_text = ' '.join((title+' . '+raw).split()[:120])
                seen.add(title_hash(title))
                records.append({
                    'source': pub.upper()[:40],
                    'published_at': e.get('published', datetime.now(timezone.utc).isoformat()),
                    'title': title, 'link': decode_google_news_url(e.get('link','')),
                    'raw_text': full_text, 'nlp_text': full_text,
                    'target_company': company, 'ticker': ticker
                })
        except: pass
        time.sleep(0.3)
    return records

def fetch_moneycontrol_rss(company, ticker, seen):
    """Moneycontrol has public RSS per company search"""
    records = []
    import urllib.parse
    query = urllib.parse.quote(company)
    url = f"https://www.moneycontrol.com/rss/buzzingstocks.xml"
    try:
        feed = feedparser.parse(url)
        co_lower = company.lower().split()[0]  # match first word e.g. "reliance"
        for e in feed.entries:
            title = e.get('title','')
            if co_lower not in title.lower(): continue
            if title_hash(title) in seen: continue
            raw = clean_html(e.get('description', e.get('summary','')))
            full_text = ' '.join((title+' . '+raw).split()[:120])
            seen.add(title_hash(title))
            records.append({
                'source': 'MONEYCONTROL',
                'published_at': e.get('published', datetime.now(timezone.utc).isoformat()),
                'title': title, 'link': decode_google_news_url(e.get('link','')),
                'raw_text': full_text, 'nlp_text': full_text,
                'target_company': company, 'ticker': ticker
            })
    except: pass
    return records

def fetch_et_rss(company, ticker, seen):
    """Economic Times markets RSS"""
    records = []
    url = "https://economictimes.indiatimes.com/markets/stocks/rssfeeds/2146842.cms"
    try:
        feed = feedparser.parse(url)
        co_lower = company.lower().split()[0]
        for e in feed.entries:
            title = e.get('title','')
            if co_lower not in title.lower(): continue
            if title_hash(title) in seen: continue
            raw = clean_html(e.get('description', e.get('summary','')))
            full_text = ' '.join((title+' . '+raw).split()[:120])
            seen.add(title_hash(title))
            records.append({
                'source': 'ECONOMIC TIMES',
                'published_at': e.get('published', datetime.now(timezone.utc).isoformat()),
                'title': title, 'link': decode_google_news_url(e.get('link','')),
                'raw_text': full_text, 'nlp_text': full_text,
                'target_company': company, 'ticker': ticker
            })
    except: pass
    return records

# ── PRE-FETCH SHARED FEEDS (once per run, not per company) ──────────────
print("🌐 Pre-fetching shared RSS feeds...")
_mc_feed = feedparser.parse("https://www.moneycontrol.com/rss/buzzingstocks.xml")
_et_feed = feedparser.parse("https://economictimes.indiatimes.com/markets/stocks/rssfeeds/2146842.cms")
print(f"   Moneycontrol: {len(_mc_feed.entries)} articles | ET Markets: {len(_et_feed.entries)} articles")

def fetch_from_prefetched(feed, source_name, company, ticker, seen):
    records = []
    co_words = [w for w in company.lower().split() if len(w)>3]
    for e in feed.entries:
        title = e.get('title','')
        if not any(w in title.lower() for w in co_words): continue
        if title_hash(title) in seen: continue
        raw = clean_html(e.get('description', e.get('summary','')))
        full_text = ' '.join((title+' . '+raw).split()[:120])
        seen.add(title_hash(title))
        records.append({
            'source': source_name,
            'published_at': e.get('published', datetime.now(timezone.utc).isoformat()),
            'title': title, 'link': decode_google_news_url(e.get('link','')),
            'raw_text': full_text, 'nlp_text': full_text,
            'target_company': company, 'ticker': ticker
        })
    return records

# ── MAIN INGESTOR LOOP ───────────────────────────────────────────────────
print(f"\n🎯 Ingesting news for {len(NIFTY_UNIVERSE)} companies...")
all_records = []
total = len(NIFTY_UNIVERSE)

for i, (ticker, company) in enumerate(NIFTY_UNIVERSE.items(), 1):
    seen_hashes = set()
    recs = []
    gn_count = mc_count = et_count = 0

    # Source 1: Google News (primary, per-company)
    try:
        gn = fetch_google_news(company, ticker, seen_hashes)
        gn_count = len(gn)
        recs.extend(gn)
    except:
        pass

    # Source 2: Moneycontrol (pre-fetched)
    try:
        mc = fetch_from_prefetched(_mc_feed, 'MONEYCONTROL', company, ticker, seen_hashes)
        mc_count = len(mc)
        recs.extend(mc)
    except:
        pass

    # Source 3: Economic Times (pre-fetched)
    try:
        et = fetch_from_prefetched(_et_feed, 'ECONOMIC TIMES', company, ticker, seen_hashes)
        et_count = len(et)
        recs.extend(et)
    except:
        pass

    all_records.extend(recs)
    print(f"   [{i:2d}/{total}] {company}: {len(recs)} articles (GN:{gn_count} | MC:{mc_count} | ET:{et_count})")
    time.sleep(0.2)


df = pd.DataFrame(all_records)
if df.empty:
    print("\n⚠️ No articles were fetched.")
    df.to_csv(os.path.join(OUT_DIR, "clean_articles.csv"), index=False)
else:
    # Core cleanup + ranking before NLP
    df['title'] = df['title'].fillna('').astype(str)
    df['raw_text'] = df['raw_text'].fillna('').astype(str)
    df['nlp_text'] = df['nlp_text'].fillna(df['raw_text']).astype(str)
    df['source'] = df['source'].fillna('').astype(str)
    df['ticker'] = df['ticker'].fillna('').astype(str)
    df['target_company'] = df['target_company'].fillna('').astype(str)
    df['published_at'] = df['published_at'].fillna('').astype(str)
    df['published_at_dt'] = pd.to_datetime(df['published_at'], errors='coerce', utc=True)
    df['content_words'] = df['nlp_text'].str.split().str.len().fillna(0).astype(int)
    df['title_sig'] = df['title'].apply(title_signature)
    df['dedupe_key'] = df['ticker'].astype(str).str.upper().str.strip() + "|" + df['title_sig'].fillna('')
    df['relevance_score'] = df.apply(compute_relevance_score, axis=1)

    # Keep strongest version of each near-duplicate headline per ticker
    df = df.sort_values(
        ['relevance_score', 'published_at_dt', 'content_words'],
        ascending=[False, False, False],
        na_position='last'
    )
    df = df.drop_duplicates(subset=['dedupe_key'], keep='first')

    # Freshness gate with fallback: if a ticker loses all recent rows, keep its top-ranked article anyway.
    cutoff = pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=14)
    def keep_group(g):
        g = g.sort_values(['relevance_score', 'published_at_dt', 'content_words'],
                          ascending=[False, False, False], na_position='last').copy()
        fresh = g[(g['published_at_dt'].notna()) & (g['published_at_dt'] >= cutoff)]
        if fresh.empty:
            fresh = g.head(1).copy()
        return fresh.head(MAX_PER_TICKER)

    df = (
        df.groupby(df['ticker'].astype(str).str.upper().str.strip(), group_keys=False)
          .apply(keep_group)
          .reset_index(drop=True)
    )

    df = df.sort_values(
        ['relevance_score', 'published_at_dt', 'content_words'],
        ascending=[False, False, False],
        na_position='last'
    ).head(MAX_TOTAL_ARTICLES).copy()

    # Final exact-title cleanup
    df = df.drop_duplicates(subset=['title', 'link'], keep='first').reset_index(drop=True)

    # Save audit + filtered corpus for downstream NLP
    df.drop(columns=['title_sig'], errors='ignore').to_csv(os.path.join(OUT_DIR, "clean_articles.csv"), index=False)

    audit = {
        "raw_articles": int(len(all_records)),
        "filtered_articles": int(len(df)),
        "unique_companies": int(df['ticker'].nunique()),
        "max_per_ticker": MAX_PER_TICKER,
        "global_cap": MAX_TOTAL_ARTICLES,
        "top_sources": df['source'].value_counts().head(10).to_dict(),
    }
    with open(os.path.join(OUT_DIR, "article_filter_audit.json"), "w") as f:
        json.dump(audit, f, indent=2)

    # Useful CSV for quick inspection if needed
    try:
        df[['ticker','target_company','source','title','link','relevance_score','published_at']].to_csv(
            os.path.join(OUT_DIR, "article_filter_audit.csv"), index=False
        )
    except Exception:
        pass

    print(f"\n✅ Ingestion complete: {len(df)} filtered articles across {df['ticker'].nunique()} companies")
    print(f"   Raw fetched: {len(all_records)}")
    print(f"   Top sources:\n{df['source'].value_counts().head(10).to_markdown()}")


🌐 Pre-fetching shared RSS feeds...
   Moneycontrol: 15 articles | ET Markets: 50 articles

🎯 Ingesting news for 50 companies...
   [ 1/50] Adani Enterprises: 49 articles (GN:48 | MC:0 | ET:1)
   [ 2/50] Adani Ports: 41 articles (GN:39 | MC:0 | ET:2)
   [ 3/50] Apollo Hospitals: 36 articles (GN:36 | MC:0 | ET:0)
   [ 4/50] Asian Paints: 49 articles (GN:49 | MC:0 | ET:0)
   [ 5/50] Axis Bank: 53 articles (GN:49 | MC:2 | ET:2)
   [ 6/50] Bajaj Auto: 56 articles (GN:52 | MC:2 | ET:2)
   [ 7/50] Bajaj Finance: 54 articles (GN:46 | MC:4 | ET:4)
   [ 8/50] Bajaj Finserv: 57 articles (GN:52 | MC:2 | ET:3)
   [ 9/50] Bharat Electronics: 35 articles (GN:35 | MC:0 | ET:0)
   [10/50] Bharti Airtel: 40 articles (GN:40 | MC:0 | ET:0)
   [11/50] Bharat Petroleum: 33 articles (GN:33 | MC:0 | ET:0)
   [12/50] Britannia: 22 articles (GN:22 | MC:0 | ET:0)
   [13/50] Cipla: 32 articles (GN:32 | MC:0 | ET:0)
   [14/50] Coal India: 45 articles (GN:38 | MC:0 | ET:7)
   [15/50] Dr Reddy: 41 articles (GN:41 | 

In [ ]:
# ============================================================
# BLOCK 5: ADVANCED SEMANTIC ENGINE
# Improvements:
#   - Per-concept thresholds (broker signals stricter than growth)
#   - Hedged language detection (reduces false positives)
#   - FinBERT + semantic must agree for full score
#   - 6 new concepts: analyst_target_raise/cut, insider_buying/selling,
#     earnings_beat/miss
# ============================================================

import pandas as pd
import numpy as np
import torch
import re
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

print("🧠 Loading models...")
device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
finbert  = pipeline("sentiment-analysis", model="ProsusAI/finbert",
                    device=0 if device=="cuda" else -1, truncation=True, max_length=512)
print(f"   Device: {device} | Models loaded ✅")

def chunk_text(text, max_words=220):
    words = str(text or "").split()
    if not words:
        return [""]
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

def batched_encode(texts, batch_size=64):
    """Encode in smaller batches to avoid OOM on Colab CPU/GPU."""
    if not texts:
        return None
    outs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        try:
            outs.append(embedder.encode(batch, convert_to_tensor=True, show_progress_bar=False))
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and torch.cuda.is_available():
                torch.cuda.empty_cache()
                outs.append(embedder.encode(batch, batch_size=max(8, batch_size//2), convert_to_tensor=True, show_progress_bar=False))
            else:
                raise
    return torch.cat(outs, dim=0) if outs else None

def finbert_batch_predict(texts, batch_size=16):
    outs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        try:
            res = finbert(batch, batch_size=max(4, batch_size), top_k=None)
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and torch.cuda.is_available():
                torch.cuda.empty_cache()
                res = finbert(batch, batch_size=4, top_k=None)
            else:
                raise
        outs.extend(res)
    return outs


# ── EXPANDED TAXONOMY (19 concepts) ─────────────────────────────────────
SIGNAL_TAXONOMY = {
    # ── BULLISH ──────────────────────────────────────────────────────────
    "expansion_capital":
        "The company raised new funding, secured capital investment, rights issue, or QIP.",
    "profit_growth":
        "Strong financial results, revenue growth, profit increase, earnings beat, PAT rise.",
    "strategic_win":
        "Major contract win, strategic partnership, joint venture, large order inflow.",
    "innovation":
        "New product launch, patent, technology breakthrough, R&D success.",
    "regulatory_ok":
        "Regulatory approval, government clearance, SEBI approval, RBI nod.",
    "broker_upgrade":
        "Brokerage upgraded the stock, raised target price, buy rating, overweight call.",
    "analyst_target_raise":
        "Analyst increased price target, raised estimates, upgraded EPS forecast.",
    "corporate_action":
        "Dividend announced, bonus shares, stock split, buyback programme.",
    "insider_buying":
        "Promoter increased stake, insider bought shares, bulk deal purchase.",
    "earnings_beat":
        "Company beat analyst estimates, PAT above expectations, revenue surprise.",
    # ── BEARISH ──────────────────────────────────────────────────────────
    "financial_distress":
        "Net loss, revenue decline, margin compression, missed earnings, debt default.",
    "leadership_exit":
        "CEO resigned, CFO stepped down, MD fired, top management exit.",
    "regulatory_crackdown":
        "SEBI action, RBI penalty, government ban, licence revocation, fine imposed.",
    "market_exit":
        "Shutting down, bankruptcy, asset sale, business closure, divestment.",
    "legal_dispute":
        "Lawsuit, court case, arbitration, fraud allegation, ED raid, CBI probe.",
    "broker_downgrade":
        "Brokerage downgraded stock, cut target price, sell rating, underweight.",
    "analyst_target_cut":
        "Analyst reduced price target, cut EPS estimates, lowered forecasts.",
    "insider_selling":
        "Promoter sold stake, insider offloaded shares, bulk deal selling.",
    "earnings_miss":
        "Missed analyst estimates, PAT below expectations, revenue disappointment.",
}

# Per-concept thresholds — stricter for concepts that overlap with general finance text
CONCEPT_THRESHOLDS = {
    "expansion_capital":    0.38,
    "profit_growth":        0.35,
    "strategic_win":        0.38,
    "innovation":           0.40,
    "regulatory_ok":        0.40,
    "broker_upgrade":       0.42,   # strict — fires on any finance text otherwise
    "analyst_target_raise": 0.42,
    "corporate_action":     0.38,
    "insider_buying":       0.40,
    "earnings_beat":        0.38,
    "financial_distress":   0.35,
    "leadership_exit":      0.40,
    "regulatory_crackdown": 0.40,
    "market_exit":          0.42,
    "legal_dispute":        0.42,   # strict — legal language is common
    "broker_downgrade":     0.42,
    "analyst_target_cut":   0.42,
    "insider_selling":      0.40,
    "earnings_miss":        0.38,
}

OPPOSING_PAIRS = [
    ("profit_growth","financial_distress"),
    ("expansion_capital","market_exit"),
    ("regulatory_ok","regulatory_crackdown"),
    ("broker_upgrade","broker_downgrade"),
    ("analyst_target_raise","analyst_target_cut"),
    ("insider_buying","insider_selling"),
    ("earnings_beat","earnings_miss"),
]

# Hedged language patterns — dampen score if article is speculative/uncertain
HEDGE_PATTERNS = re.compile(
    r'\b(may|might|could|expected to|likely|possible|analysts predict|'
    r'if approved|pending|subject to|reportedly|sources say|rumour|'
    r'unconfirmed|alleged|could face)\b', re.IGNORECASE
)


def finbert_signed_score(result):
    """Convert FinBERT output to a signed score in [-1, 1]."""
    try:
        if isinstance(result, list):
            scores = {str(x["label"]).lower(): float(x["score"]) for x in result if isinstance(x, dict) and "label" in x and "score" in x}
        elif isinstance(result, dict):
            scores = {str(result.get("label","")).lower(): float(result.get("score", 0.0))}
        else:
            return 0.0
        pos = scores.get("positive", 0.0)
        neg = scores.get("negative", 0.0)
        neu = scores.get("neutral", 0.0)
        # Neutral should dampen the magnitude slightly rather than collapse to zero
        return float((pos - neg) * max(0.35, 1.0 - neu * 0.5))
    except Exception:
        return 0.0

def pick_finbert_label(result):
    try:
        if isinstance(result, list):
            best = max(result, key=lambda x: float(x.get("score", 0.0)))
            return str(best.get("label", "neutral")).lower(), float(best.get("score", 0.0))
        if isinstance(result, dict):
            return str(result.get("label", "neutral")).lower(), float(result.get("score", 0.0))
    except Exception:
        pass
    return "neutral", 0.0

def extract_signals():
    df = pd.read_csv("/tmp/agentic_sector/clean_articles.csv").fillna("")
    print(f"🚀 Semantic extraction on {len(df)} articles...")

    concepts = list(SIGNAL_TAXONOMY.keys())
    concept_texts = list(SIGNAL_TAXONOMY.values())
    concept_embs = embedder.encode(concept_texts, convert_to_tensor=True, show_progress_bar=False)

    # Use cleaned article text with chunked context (up to ~440 words in the first two chunks)
    snippets = [' '.join(chunk_text(str(r['nlp_text']), max_words=220)[:2]) for _, r in df.iterrows()]

    print("   → Encoding article vectors in batches...")
    art_embs = batched_encode(snippets, batch_size=64 if device == 'cuda' else 32)
    cosine_scores = util.cos_sim(art_embs, concept_embs).cpu().numpy() if art_embs is not None else np.zeros((len(snippets), len(concepts)))

    print("   → Running FinBERT sentiment validation in batches...")
    finbert_results = finbert_batch_predict(snippets, batch_size=16)

    results = []
    for i, row in df.iterrows():
        found = {}
        reasons = []
        text = snippets[i] if i < len(snippets) else ""

        # Hedge penalty: speculative language reduces signal strength
        hedge_count = len(HEDGE_PATTERNS.findall(text))
        hedge_penalty = max(0.5, 1.0 - hedge_count * 0.08)

        # Semantic concept matching
        for j, concept in enumerate(concepts):
            score = float(cosine_scores[i][j])
            thresh = CONCEPT_THRESHOLDS.get(concept, 0.38)
            if score > thresh:
                weight = score * 10 * hedge_penalty
                found[f"{concept}_weight"] = weight
                reasons.append(f"📡 Semantic Match: {concept.replace('_',' ').title()} ({int(score*100)}%) hedge×{hedge_penalty:.2f}")

        # Resolve opposing concepts: keep the stronger one
        for bull, bear in OPPOSING_PAIRS:
            bk, nk = f"{bull}_weight", f"{bear}_weight"
            if bk in found and nk in found:
                if found[bk] >= found[nk]:
                    del found[nk]
                else:
                    del found[bk]

        fb = finbert_results[i] if i < len(finbert_results) else {"label": "neutral", "score": 0.0}
        fb_label, fb_conf = pick_finbert_label(fb)
        fb_signed = finbert_signed_score(fb)

        semantic_total = float(sum(found.values()))
        if semantic_total > 0:
            # Adaptive multiplier: positive FinBERT amplifies, negative flips, neutral dampens mildly.
            if fb_label == "positive":
                multiplier = 1.0 + max(0.0, fb_conf - 0.50) * 1.4
                reasons.append(f"🧠 FinBERT Bullish ({fb_conf:.0%})")
            elif fb_label == "negative":
                multiplier = -(1.0 + max(0.0, fb_conf - 0.50) * 1.4)
                reasons.append(f"🚨 FinBERT Bearish ({fb_conf:.0%})")
            else:
                multiplier = 0.65
                reasons.append("⚪ FinBERT Neutral — dampened")
            for k in list(found.keys()):
                found[k] *= multiplier
            sentiment_score = semantic_total * multiplier
        else:
            # Fallback so sentiment never becomes zero for every article
            sentiment_score = fb_signed * 2.5
            if abs(sentiment_score) < 0.15:
                sentiment_score = 0.15 if fb_signed >= 0 else -0.15
            reasons.append("🧭 Semantic fallback not triggered — using FinBERT only")

        # Extra adaptive boost if the article is both highly relevant and fresh
        recency_boost = 1.0
        try:
            pub = pd.to_datetime(row.get("published_at", ""), errors="coerce", utc=True)
            if pd.notna(pub):
                age_hours = max((pd.Timestamp.now(tz="UTC") - pub).total_seconds() / 3600.0, 0.0)
                recency_boost = float(np.exp(-age_hours / 72.0) * 1.25 + 0.75)
        except Exception:
            pass
        sentiment_score *= recency_boost

        entry = {
            "row_index": i,
            "title": row["title"],
            "link": row.get("link", ""),
            "human_reasons": reasons,
            "sentiment": round(float(sentiment_score), 4),
            "sentiment_score": round(float(sentiment_score), 4),
            "semantic_score": round(float(semantic_total), 4),
            "finbert_label": fb_label,
            "finbert_conf": round(float(fb_conf), 3),
            "finbert_signed": round(float(fb_signed), 4),
        }
        entry.update(found)
        results.append(entry)

    out = pd.DataFrame(results).fillna(0)
    out.to_csv("/tmp/agentic_sector/opportunity_signals.csv", index=False)
    bull = len(out[out[[c for c in out.columns if c.endswith('_weight')]].sum(axis=1) > 0])
    bear = len(out[out[[c for c in out.columns if c.endswith('_weight')]].sum(axis=1) < 0])
    print(f"✅ Extraction done. Bullish signals: {bull} | Bearish: {bear} | Neutral: {len(out)-bull-bear}")

extract_signals()


🧠 Loading models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   Device: cpu | Models loaded ✅
🚀 Semantic extraction on 905 articles...
   → Encoding article vectors...


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

   → Running FinBERT sentiment validation...
✅ Extraction done. Bullish signals: 375 | Bearish: 103 | Neutral: 427


In [ ]:
# ============================================================
# BLOCK 6: ADAPTIVE MARKET-RELATIVE SCORING ENGINE
# ============================================================
import pandas as pd, numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from scipy.stats import percentileofscore
from datetime import datetime, timezone, timedelta
import ast

def run_adaptive_z_score_engine():
    df_sig = pd.read_csv("/tmp/agentic_sector/opportunity_signals.csv").fillna(0)
    df_clean = pd.read_csv("/tmp/agentic_sector/clean_articles.csv")
    df = df_sig.merge(df_clean[['title','published_at','source','target_company']], on='title', how='left')

    feat_cols = [c for c in df.columns if c.endswith('_weight')]
    if not feat_cols:
        print("⚠️ No signal columns found."); return
    X = df[feat_cols].fillna(0)

    # ── Adaptive contamination ────────────────────────────────
    contamination = max(0.04, min(0.15, 15 / max(len(df), 1)))
    print(f"🌪️ Anomaly scan  (n={len(df)}, contamination={contamination:.3f})")
    df['is_anomaly'] = IsolationForest(contamination=contamination, random_state=42).fit_predict(X)

    # ── Logistic weight calibration ───────────────────────────
    sentiment_src = df['sentiment_score'] if 'sentiment_score' in df.columns else df.get('sentiment', pd.Series(dtype=float))
    df['target_bullish'] = (pd.to_numeric(sentiment_src, errors='coerce').fillna(0) > 0.15).astype(int)
    Y = df['target_bullish']
    fallback_w = {
        "expansion_capital_weight":3.0,"profit_growth_weight":4.0,
        "strategic_win_weight":2.5,"innovation_weight":2.0,
        "regulatory_ok_weight":2.5,"broker_upgrade_weight":3.5,
        "analyst_target_raise_weight":3.5,"corporate_action_weight":2.0,
        "insider_buying_weight":3.0,"earnings_beat_weight":4.0,
        "financial_distress_weight":4.0,"leadership_exit_weight":3.0,
        "regulatory_crackdown_weight":3.5,"market_exit_weight":2.5,
        "legal_dispute_weight":3.0,"broker_downgrade_weight":3.5,
        "analyst_target_cut_weight":3.5,"insider_selling_weight":3.0,
        "earnings_miss_weight":4.0,
    }
    calc_w = {k:v for k,v in fallback_w.items() if k in feat_cols}
    if Y.sum() >= 3 and (len(Y)-Y.sum()) >= 3 and len(calc_w) >= 3:
        try:
            Xs = pd.DataFrame(StandardScaler().fit_transform(X[list(calc_w.keys())]), columns=list(calc_w.keys()))
            lr = LogisticRegression(
                penalty='elasticnet', solver='saga', l1_ratio=0.35,
                class_weight='balanced', max_iter=2000, random_state=42
            )
            lr.fit(Xs, Y)
            cw = {k:abs(v) for k, v in zip(Xs.columns, lr.coef_[0]) if abs(v) > 0.01}
            if cw:
                calc_w = cw
                print("   ✅ Elastic-net weights calibrated from data")
        except Exception as e:
            print(f"   ⚠️ Weight calibration fallback: {e}")

    # ── Authority multipliers (partial match) ─────────────────
    SOURCE_RULES = [
        (["bloomberg","reuters"],1.25),
        (["cnbc","cnbc-tv18"],1.20),
        (["mint","livemint"],1.15),
        (["economic times","et markets"],1.12),
        (["business standard"],1.10),
        (["moneycontrol"],1.08),
        (["financial express","bq prime"],1.06),
        (["businessline","the hindu"],1.05),
    ]
    def authority(s):
        s=str(s).lower()
        for keys,m in SOURCE_RULES:
            if any(k in s for k in keys): return m
        return 1.0

    # ── Dynamic decay by news velocity (recent 3d burst vs 14d baseline) ───────────────────────
    now_utc = datetime.now(timezone.utc)
    df['published_at_dt'] = pd.to_datetime(df.get('published_at'), errors='coerce', utc=True)
    cutoff_3d  = pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=3)
    cutoff_14d = pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=14)
    counts_14 = df.groupby('target_company')['published_at_dt'].transform(lambda s: int(((s >= cutoff_14d) & s.notna()).sum()))
    counts_3  = df.groupby('target_company')['published_at_dt'].transform(lambda s: int(((s >= cutoff_3d) & s.notna()).sum()))
    df['news_velocity'] = ((counts_3 + 1) / (counts_14 + 1)).clip(0.0, 2.0)
    df['decay_lambda'] = np.clip(0.025 + (counts_14 / 220.0) + (1.0 - df['news_velocity']) * 0.04, 0.025, 0.12)

    raw_scores = []
    for _, row in df.iterrows():
        base = sum(row.get(sig,0)*w for sig,w in calc_w.items())
        base *= authority(row.get('source',''))
        try:
            pub = pd.to_datetime(row['published_at'])
            if pub.tzinfo is None: pub = pub.replace(tzinfo=timezone.utc)
            days_old = max(0, (now_utc-pub).days)
            base *= np.exp(-float(row['decay_lambda'])*days_old)
        except: pass
        raw_scores.append(base)
    df['raw_score'] = raw_scores

    # ── Adaptive percentile thresholds ────────────────────────
    active_abs = df.loc[df['raw_score'].abs() > 1e-9, 'raw_score'].abs()
    if len(active_abs) < 5:
        all_abs = np.array([1.0, 1.5, 2.0, 2.5])
    else:
        # winsorize tiny noise and preserve adaptive spread
        cap = np.percentile(active_abs, 95)
        all_abs = np.clip(active_abs.values, 1e-3, cap)
    mean_mag = float(np.mean(all_abs)); std_mag = max(float(np.std(all_abs)), 1e-6)
    T_HIGH   = float(np.percentile(all_abs, 85))
    T_MEDIUM = float(np.percentile(all_abs, 55))
    T_WATCH  = float(np.percentile(all_abs, 25))
    max_abs  = float(max(np.max(all_abs), 1e-3)) + 1e-9

    regime = "VOLATILE" if std_mag > mean_mag*0.8 else ("TRENDING" if mean_mag > std_mag else "QUIET")
    print(f"📊 Thresholds HIGH≥{T_HIGH:.4f} | MED≥{T_MEDIUM:.4f} | WATCH≥{T_WATCH:.4f} | Regime: {regime}")

    alerts = []
    for _, row in df.iterrows():
        raw = row['raw_score']
        # Safe reason parsing
        hr = row.get('human_reasons', '[]')
        try:    reasons = ast.literal_eval(str(hr)) if str(hr).startswith('[') else [str(hr)]
        except: reasons = [str(hr)] if str(hr) else []

        if raw == 0:
            fs = 0.0
        else:
            sign = 1 if raw>0 else -1
            z = (abs(raw)-mean_mag)/std_mag
            fs = max(-10.0, min(10.0, round((2.5+z*2.5)*sign, 1)))
            if row.get('is_anomaly')==-1 and abs(fs)>(T_MEDIUM/max_abs*8):
                fs = max(-10.0,min(10.0,fs*1.3))
                reasons.append("🚨 ALGORITHMIC BREAKOUT DETECTED")

        abs_fs = abs(fs)
        norm_fs = abs_fs / max(T_HIGH/max_abs*10, 1e-9)

        if abs_fs == 0:   status = "📰 GENERAL COVERAGE"
        elif fs > 0:
            if abs_fs >= T_HIGH/max_abs*10:   status = "🚀 HIGH PRIORITY BUY"
            elif abs_fs >= T_MEDIUM/max_abs*10: status = "📈 POTENTIAL BUY"
            elif abs_fs >= T_WATCH/max_abs*10:  status = "👀 WATCHLIST (BULL)"
            else:                               status = "📰 GENERAL COVERAGE"
        else:
            if abs_fs >= T_HIGH/max_abs*10:   status = "🚨 HIGH PRIORITY RISK"
            elif abs_fs >= T_MEDIUM/max_abs*10: status = "📉 POTENTIAL LOSS"
            elif abs_fs >= T_WATCH/max_abs*10:  status = "⚠️ WATCHLIST (BEAR)"
            else:                               status = "📰 GENERAL COVERAGE"

        alerts.append({"title":row['title'], "score":fs, "status":status,
                       "color":"#10B981", "reasons":str(reasons),
                       "link":row.get('link',''), "regime":regime})

    df_out = pd.DataFrame(alerts).sort_values("score", ascending=False)
    df_out.to_csv("/tmp/agentic_sector/final_ml_alerts.csv", index=False)
    print(f"\n📊 Distribution:\n{df_out['status'].value_counts().to_markdown()}")
    print(f"✅ Block 6 complete. Regime: {regime}")

run_adaptive_z_score_engine()


🌪️ Anomaly scan  (n=907, contamination=0.040)
   ✅ Logistic weights calibrated from data
📊 Thresholds HIGH≥0.8411 | MED≥0.0011 | WATCH≥0.0010 | Regime: VOLATILE

📊 Distribution:
| status                |   count |
|:----------------------|--------:|
| 📰 GENERAL COVERAGE   |     427 |
| 🚀 HIGH PRIORITY BUY  |     377 |
| 🚨 HIGH PRIORITY RISK |     103 |
✅ Block 6 complete. Regime: VOLATILE


In [ ]:
# ============================================================
# BLOCK 7: ADAPTIVE QUANT ENGINE
# FIXED:
#   - No re-install (done in Block 0)
#   - yfinance MultiIndex/auto_adjust/group_by fully safe
#   - Single-download-with-cache via _yf_download/_bulk_get
#   - normalize_special_tickers defined once
#   - RSI divergence + ADX scoring improved
# ============================================================
import pandas as pd, numpy as np, pandas_ta as ta
import os, warnings, requests, json
from scipy.stats import zscore as scipy_zscore
warnings.filterwarnings('ignore')

OUT_DIR = "/tmp/agentic_sector"
os.makedirs(OUT_DIR, exist_ok=True)


# ── yfinance safe downloader (shared across all blocks) ─────────────
# Fixes deprecated group_by='ticker', MultiIndex pandas bug, auto_adjust schema
def _yf_download(tickers_list, period="6mo", progress=False):
    """Single safe download entry point. Returns dict: {ticker: DataFrame}."""
    import yfinance as yf, pandas as pd
    single = isinstance(tickers_list, str)
    tickers_list = [tickers_list] if single else list(tickers_list)
    # yfinance ≥0.2.38 uses multi_level_index; older uses group_by
    import inspect
    dl_sig = inspect.signature(yf.download).parameters
    kwargs = dict(period=period, progress=progress, auto_adjust=True)
    if "multi_level_index" in dl_sig:
        kwargs["multi_level_index"] = False   # flat columns, no MultiIndex
    else:
        kwargs["group_by"] = "ticker"         # old API
    raw = yf.download(tickers_list, **kwargs)
    if raw.empty:
        return {}
    result = {}
    if len(tickers_list) == 1:
        tk = tickers_list[0]
        result[tk] = raw.dropna(subset=["Close"]) if "Close" in raw.columns else raw
    else:
        if isinstance(raw.columns, pd.MultiIndex):
            for tk in tickers_list:
                try:
                    h = raw.xs(tk, axis=1, level=1).copy().dropna(subset=["Close"])
                    if not h.empty: result[tk] = h
                except: pass
        else:
            # flat columns: happens with multi_level_index=False when n=1, else ticker-prefixed
            for tk in tickers_list:
                try:
                    cols = {c.split("_")[1] if "_" in c else c: raw[c] for c in raw.columns if c.endswith(f"_{tk}") or c.endswith(f" {tk}")}
                    if cols:
                        h = pd.DataFrame(cols).dropna(subset=["Close"])
                        if not h.empty: result[tk] = h
                    elif tk in raw.columns.get_level_values(0) if isinstance(raw.columns, pd.MultiIndex) else []:
                        h = raw[tk].dropna(subset=["Close"])
                        if not h.empty: result[tk] = h
                    else:
                        # try direct key
                        if tk in raw.columns:
                            result[tk] = raw[[tk]].rename(columns={tk:"Close"})
                except: pass
            # Final fallback: try xs
            if not result:
                for tk in tickers_list:
                    try:
                        h = raw.xs(tk, axis=1, level=1).copy().dropna(subset=["Close"])
                        if not h.empty: result[tk] = h
                    except: pass
    return result

_SHARED_PRICE_CACHE = {}   # {(ticker,period): DataFrame} — avoids re-downloading

def _get_hist(ticker, period="6mo", min_rows=45):
    """Get single ticker history. Uses cache to avoid re-downloads."""
    key = (ticker, period)
    if key in _SHARED_PRICE_CACHE:
        return _SHARED_PRICE_CACHE[key]
    d = _yf_download([ticker], period=period)
    h = d.get(ticker, __import__("pandas").DataFrame())
    _SHARED_PRICE_CACHE[key] = h
    return h

def _bulk_get(tickers_list, period="6mo"):
    """Bulk download with cache. Returns {ticker: DataFrame}."""
    missing = [t for t in tickers_list if (t,period) not in _SHARED_PRICE_CACHE]
    if missing:
        dl = _yf_download(missing, period=period)
        for tk, h in dl.items():
            _SHARED_PRICE_CACHE[(tk,period)] = h
        for tk in missing:
            if (tk,period) not in _SHARED_PRICE_CACHE:
                _SHARED_PRICE_CACHE[(tk,period)] = __import__("pandas").DataFrame()
    return {tk: _SHARED_PRICE_CACHE[(tk,period)] for tk in tickers_list}


def robust_zscore(arr):
    arr = np.asarray(arr, dtype=float)
    med = np.median(arr)
    mad = np.median(np.abs(arr - med))
    if mad < 1e-9:
        std = np.std(arr) + 1e-9
        return (arr - np.mean(arr)) / std
    return (arr - med) / (1.4826 * mad + 1e-9)

# ── UNIVERSE ─────────────────────────────────────────────────
NIFTY50_HARDCODED = {
    "ADANIENT.NS":"Adani Enterprises","ADANIPORTS.NS":"Adani Ports",
    "APOLLOHOSP.NS":"Apollo Hospitals","ASIANPAINT.NS":"Asian Paints",
    "AXISBANK.NS":"Axis Bank","BAJAJ-AUTO.NS":"Bajaj Auto",
    "BAJFINANCE.NS":"Bajaj Finance","BAJAJFINSV.NS":"Bajaj Finserv",
    "BEL.NS":"Bharat Electronics","BHARTIARTL.NS":"Bharti Airtel",
    "BPCL.NS":"Bharat Petroleum","BRITANNIA.NS":"Britannia",
    "CIPLA.NS":"Cipla","COALINDIA.NS":"Coal India","DRREDDY.NS":"Dr Reddy",
    "EICHERMOT.NS":"Eicher Motors","GRASIM.NS":"Grasim",
    "HCLTECH.NS":"HCL Tech","HDFCBANK.NS":"HDFC Bank",
    "HDFCLIFE.NS":"HDFC Life","HEROMOTOCO.NS":"Hero MotoCorp",
    "HINDALCO.NS":"Hindalco","HINDUNILVR.NS":"Hindustan Unilever",
    "ICICIBANK.NS":"ICICI Bank","INDUSINDBK.NS":"IndusInd Bank",
    "INFY.NS":"Infosys","ITC.NS":"ITC","JSWSTEEL.NS":"JSW Steel",
    "KOTAKBANK.NS":"Kotak Mahindra Bank","LT.NS":"Larsen & Toubro",
    "M&M.NS":"Mahindra & Mahindra","MARUTI.NS":"Maruti Suzuki",
    "NESTLEIND.NS":"Nestle India","NTPC.NS":"NTPC",
    "ONGC.NS":"ONGC","POWERGRID.NS":"Power Grid",
    "RELIANCE.NS":"Reliance Industries","SBILIFE.NS":"SBI Life",
    "SBIN.NS":"State Bank India","SHRIRAMFIN.NS":"Shriram Finance",
    "SUNPHARMA.NS":"Sun Pharma","TATACONSUM.NS":"Tata Consumer",
    "TMPV.NS":"Tata Motors Passenger Vehicles",
    "TMCV.NS":"Tata Motors Commercial Vehicles","TATASTEEL.NS":"Tata Steel",
    "TCS.NS":"TCS","TECHM.NS":"Tech Mahindra","TITAN.NS":"Titan",
    "TRENT.NS":"Trent","ULTRACEMCO.NS":"UltraTech Cement","WIPRO.NS":"Wipro",
}

def get_nifty50_universe():
    """3-tier: NSE API → NSE CSV → hardcoded fallback."""
    # Tier 1: NSE API
    try:
        import requests as _req
        hdrs = {'User-Agent':'Mozilla/5.0','Accept':'application/json','Referer':'https://www.nseindia.com'}
        s = _req.Session()
        s.get('https://www.nseindia.com', headers=hdrs, timeout=5)
        r = s.get('https://www.nseindia.com/api/equity-stockIndices?index=NIFTY%2050', headers=hdrs, timeout=8)
        if r.status_code == 200:
            stocks = r.json().get('data',[])
            u = {f"{s['symbol']}.NS": (s.get('meta',{}) or {}).get('companyName', s['symbol'])
                 for s in stocks if s.get('symbol') and s['symbol'] != 'NIFTY 50'}
            if "TATAMOTORS.NS" in u:
                u.pop("TATAMOTORS.NS", None)
                u["TMPV.NS"] = "Tata Motors Passenger Vehicles"
                u["TMCV.NS"] = "Tata Motors Commercial Vehicles"
            if len(u) >= 40:
                print(f"✅ NSE API: {len(u)} stocks"); return u
    except Exception as e:
        print(f"   NSE API: {e}")
    # Tier 2: NSE CSV
    try:
        from io import StringIO
        import requests as _req
        r = _req.get('https://archives.nseindia.com/content/indices/ind_nifty50list.csv',
                     timeout=8, headers={'User-Agent':'Mozilla/5.0'})
        if r.status_code == 200:
            df = pd.read_csv(StringIO(r.text))
            sc = next((c for c in df.columns if 'symbol' in c.lower()), None)
            nc = next((c for c in df.columns if 'company' in c.lower()), None)
            if sc:
                u = {f"{row[sc].strip()}.NS": str(row[nc]).strip() if nc else row[sc].strip() for _,row in df.iterrows()}
                if "TATAMOTORS.NS" in u:
                    u.pop("TATAMOTORS.NS", None)
                    u["TMPV.NS"] = "Tata Motors Passenger Vehicles"
                    u["TMCV.NS"] = "Tata Motors Commercial Vehicles"
                if len(u) >= 40:
                    print(f"✅ NSE CSV: {len(u)} stocks"); return u
    except Exception as e:
        print(f"   NSE CSV: {e}")
    print("⚠️ Using built-in universe"); return NIFTY50_HARDCODED.copy()

def detect_rsi_divergence(hist, rsi_col):
    if rsi_col not in hist.columns or len(hist) < 20: return 0.0
    r = hist.tail(20)
    p = r['Close'].values; rsi = r[rsi_col].values
    pl = (np.argmin(p[:10]), np.argmin(p[10:])+10)
    ph = (np.argmax(p[:10]), np.argmax(p[10:])+10)
    if p[pl[1]] < p[pl[0]] and rsi[pl[1]] > rsi[pl[0]] + 2: return +2.0
    if p[ph[1]] > p[ph[0]] and rsi[ph[1]] < rsi[ph[0]] - 2: return -2.0
    return 0.0

def run_adaptive_quant():
    print("📈 Adaptive Quant Engine v5 (fixed)...")
    universe = get_nifty50_universe()
    tickers  = list(universe.keys())
    BENCHMARK = "^NSEI"

    print(f"📡 Bulk-downloading 6mo data for {len(tickers)+1} symbols (cached for all blocks)...")
    all_dl = _bulk_get(tickers + [BENCHMARK], period="6mo")

    bm_hist = all_dl.get(BENCHMARK, pd.DataFrame())
    if len(bm_hist) < 21:
        bm_hist = _get_hist(BENCHMARK, "6mo")
    bm_ret20 = float(bm_hist['Close'].pct_change(20).iloc[-1]) if len(bm_hist)>20 else 0.0

    raw_results = []
    for ticker in tickers:
        h = all_dl.get(ticker, pd.DataFrame())
        if len(h) < 45:
            h2 = _get_hist(ticker, "6mo")
            if len(h2) >= 45: h = h2
        if len(h) < 45: continue

        # Compute indicators safely
        try: h.ta.ema(length=20, append=True)
        except: pass
        try: h.ta.ema(length=50, append=True)
        except: pass
        try: h.ta.sma(length=20, append=True)
        except: pass
        try: h.ta.macd(fast=12, slow=26, signal=9, append=True)
        except: pass
        try: h.ta.rsi(length=14, append=True)
        except: pass
        try: h.ta.atr(length=14, append=True)
        except: pass
        try: h.ta.adx(length=14, append=True)
        except: pass
        try: h.ta.bbands(length=20, std=2, append=True)
        except: pass
        try: h.ta.obv(append=True)
        except: pass

        last  = h.iloc[-1]
        close = float(last['Close'])

        def gcol(prefix, fallback=0.0):
            c = next((x for x in h.columns if str(x).upper().startswith(prefix.upper())), None)
            try: return float(last[c]) if c and pd.notna(last[c]) else fallback
            except: return fallback

        atr   = gcol('ATR', close*0.02)
        rsi   = gcol('RSI', 50)
        macdh = gcol('MACDH', 0)
        ema20 = gcol('EMA_20', close)
        ema50 = gcol('EMA_50', close)
        sma20 = gcol('SMA_20', close)
        adx   = gcol('ADX_', 20)
        bb_up = gcol('BBU_', close*1.05)
        bb_lo = gcol('BBL_', close*0.95)

        avg_vol = float(h['Volume'].rolling(20).mean().iloc[-1]) if 'Volume' in h.columns else 1e-9
        rvol    = float(last.get('Volume', avg_vol)) / (avg_vol + 1e-9)
        spread  = float(h['Close'].pct_change(20).iloc[-1]) - bm_ret20 if len(h)>20 else 0.0

        score = 0.0
        # Trend
        if   close > ema20 > ema50: score += 2.0
        elif close < ema20 < ema50: score -= 2.0
        # MACD
        score += 2.0 if macdh > 0 else -2.0
        # RS vs benchmark (ADX-confidence weighted)
        adx_conf = min(1.5, adx/25) if adx > 0 else 1.0
        score += float(np.clip(spread*30, -3.0, 3.0)) * adx_conf
        # RSI divergence
        rsi_col = next((c for c in h.columns if str(c).upper().startswith('RSI')), None)
        score  += detect_rsi_divergence(h, rsi_col) if rsi_col else 0.0
        # Bollinger
        if close > bb_up: score -= 0.5
        elif close < bb_lo: score += 0.5
        # Overextension
        dist = (close - sma20) / (atr + 1e-9)
        if abs(dist) > 3.0: score *= 0.5
        # Volume (additive, not multiplicative, so it does not dominate the score)
        if rvol > 1.8:
            score += 1.5
        elif rvol < 0.5:
            score -= 1.0
        else:
            score += float(np.clip((rvol - 1.0) * 1.0, -0.5, 0.8))

        raw_results.append({
            "ticker": ticker, "company_name": universe[ticker],
            "raw_score": score, "RVOL": round(rvol,2),
            "RSI": round(rsi,1), "ADX": round(adx,1),
            "Close": round(close,2), "ATR": round(atr,2),
            "Stop Loss (2x ATR)": round(close-(2*atr),2),
            "Relative_Spread": round(spread*100,2),
            "RSI_Divergence": detect_rsi_divergence(h, rsi_col) if rsi_col else 0.0,
        })

    if not raw_results:
        print("⚠️ No quant results — check internet connectivity in Colab."); return

    df_raw  = pd.DataFrame(raw_results)
    scores  = df_raw['raw_score'].values
    med     = np.median(scores)
    abs_s   = np.abs(scores - med)
    P_STRONG = np.percentile(abs_s, 80)
    P_ACTION = np.percentile(abs_s, 50)

    zscores = robust_zscore(scores)
    quant_rows = []
    for idx, r in df_raw.iterrows():
        s = r['raw_score']; z = float(zscores[idx])
        final = max(-10.0, min(10.0, round(z*3.5, 1)))
        abs_r = abs(s)
        if   abs_r >= P_STRONG: status = "🔥 STRONG BUY"     if s>0 else "🩸 STRONG SELL"
        elif abs_r >= P_ACTION: status = "🟢 ACCUMULATE"     if s>0 else "🔴 DISTRIBUTE"
        else:                   status = "⚪ CHOP / NEUTRAL"
        quant_rows.append({
            "ticker":r["ticker"],"company_name":r["company_name"],
            "quant_score":final,"status":status,
            "Relative Outperformance":f"{r['Relative_Spread']}%",
            "Stop Loss (2x ATR)":r["Stop Loss (2x ATR)"],
            "ATR":r["ATR"],"RVOL":r["RVOL"],"RSI":r["RSI"],
            "ADX":r["ADX"],"RSI_Divergence":r["RSI_Divergence"],"Close":r["Close"],
        })

    df_q = pd.DataFrame(quant_rows).sort_values("quant_score", ascending=False)
    df_q.to_csv(os.path.join(OUT_DIR,"institutional_quant_signals.csv"), index=False)
    print(f"\n✅ Quant engine done.\n{df_q['status'].value_counts().to_markdown()}")
    print(df_q[['ticker','quant_score','status','RSI','ADX']].head(12).to_markdown(index=False))

run_adaptive_quant()


📈 Adaptive Quant Engine v5 (fixed)...
✅ NSE API: 50 stocks
📡 Bulk-downloading 6mo data for 51 symbols (cached for all blocks)...

✅ Quant engine done.
| status            |   count |
|:------------------|--------:|
| ⚪ CHOP / NEUTRAL |      25 |
| 🟢 ACCUMULATE     |       9 |
| 🔥 STRONG BUY     |       6 |
| 🔴 DISTRIBUTE     |       6 |
| 🩸 STRONG SELL    |       4 |
| ticker        |   quant_score | status        |   RSI |   ADX |
|:--------------|--------------:|:--------------|------:|------:|
| ADANIENT.NS   |           5.8 | 🔥 STRONG BUY |  74.6 |  42.8 |
| GRASIM.NS     |           5.5 | 🔥 STRONG BUY |  65.9 |  16.8 |
| HINDUNILVR.NS |           4.7 | 🔥 STRONG BUY |  60.1 |  26.1 |
| NESTLEIND.NS  |           4.6 | 🔥 STRONG BUY |  81   |  48.7 |
| RELIANCE.NS   |           4.5 | 🔥 STRONG BUY |  66.4 |  22.2 |
| ADANIPORTS.NS |           4.4 | 🔥 STRONG BUY |  74.6 |  26.9 |
| M&M.NS        |           4.1 | 🟢 ACCUMULATE |  54.4 |  12.7 |
| TATACONSUM.NS |           4   | 🟢 ACCUMUL

In [ ]:
# ============================================================
# BLOCK 7.1: TIME-SERIES FORECASTING TOURNAMENT
# FIXED:
#   - Uses _bulk_get cache — NO re-download
#   - TES seasonal fallback if seasonal_periods fails
#   - Robust per-ticker error handling
# ============================================================
import pandas as pd, numpy as np, os, warnings
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error
warnings.filterwarnings("ignore")

# _yf_download, _SHARED_PRICE_CACHE, _bulk_get defined in Block 7 above

def run_forecasting_tournament():
    print("🔮 Forecasting Tournament (uses cached price data)...")
    OUT_DIR = "/tmp/agentic_sector"
    quant_file = os.path.join(OUT_DIR, "institutional_quant_signals.csv")
    if not os.path.exists(quant_file):
        print("⚠️ Run Block 7 first."); return

    df_quant  = pd.read_csv(quant_file)
    tickers   = df_quant["ticker"].tolist()
    co_names  = dict(zip(df_quant["ticker"], df_quant["company_name"]))

    # Use cache — no new download
    print(f"📊 Running tournament for {len(tickers)} tickers (from cache)...")
    all_data  = _bulk_get(tickers, period="6mo")

    def fit_model(name, data, steps=5):
        try:
            if name == "MA":
                return ARIMA(data, order=(0,0,2)).fit().forecast(steps=steps)
            elif name == "ARIMA":
                return ARIMA(data, order=(1,1,1)).fit().forecast(steps=steps)
            elif name == "SES":
                return ExponentialSmoothing(data, trend=None, seasonal=None).fit().forecast(steps)
            elif name == "DES":
                return ExponentialSmoothing(data, trend="add", seasonal=None).fit().forecast(steps)
            elif name == "TES":
                # Try TES; fall back to DES if seasonal fails
                try:
                    return ExponentialSmoothing(data, trend="add", seasonal="add", seasonal_periods=22).fit().forecast(steps)
                except:
                    return ExponentialSmoothing(data, trend="add", seasonal=None).fit().forecast(steps)
        except:
            return None

    results = []
    for ticker in tickers:
        h = all_data.get(ticker, pd.DataFrame())
        if len(h) < 60: continue
        prices  = h["Close"].values.astype(float)
        cp      = prices[-1]
        train   = prices[:-5]; test = prices[-5:]

        best_name, best_rmse = "MA", float("inf")
        for name in ["MA","ARIMA","SES","DES","TES"]:
            preds = fit_model(name, train)
            if preds is not None and len(preds) == 5:
                rmse = float(np.sqrt(mean_squared_error(test, preds)))
                if rmse < best_rmse:
                    best_rmse = rmse; best_name = name

        fc = fit_model(best_name, prices)
        if fc is None: fc = [cp]*5
        d1,d2,d3,d4,d5 = [float(x) for x in fc[:5]]
        exp_ret = (d5 - cp) / cp * 100
        results.append({
            "ticker": ticker, "Company": co_names.get(ticker,ticker),
            "Winning Model": best_name, "Model RMSE": round(best_rmse,2),
            "Current Price": round(cp,2),
            "Day 1":round(d1,2),"Day 2":round(d2,2),"Day 3":round(d3,2),
            "Day 4":round(d4,2),"Day 5":round(d5,2),
            "Expected Move": round(exp_ret,2)
        })

    df_fc = pd.DataFrame(results).sort_values("Expected Move", ascending=False)
    df_fc.to_csv(os.path.join(OUT_DIR,"predictive_forecasts.csv"), index=False)
    print(f"\n✅ Forecasting done. {len(df_fc)} tickers.")
    df_fc["Expected Move"] = df_fc["Expected Move"].astype(str) + "%"
    print(df_fc[["ticker","Winning Model","Current Price","Day 1","Day 5","Expected Move"]].head(10).to_markdown(index=False))

run_forecasting_tournament()


🔮 Forecasting Tournament (uses cached price data)...
📊 Running tournament for 50 tickers (from cache)...

✅ Forecasting done. 50 tickers.
| ticker        | Winning Model   |   Current Price |    Day 1 |    Day 5 | Expected Move   |
|:--------------|:----------------|----------------:|---------:|---------:|:----------------|
| ADANIPORTS.NS | DES             |          1725   |  1744.54 |  1805.96 | 4.69%           |
| TECHM.NS      | MA              |          1452.2 |  1448.89 |  1509.22 | 3.93%           |
| NESTLEIND.NS  | ARIMA           |          1477.8 |  1483.48 |  1502.44 | 1.67%           |
| COALINDIA.NS  | TES             |           472.6 |   475.48 |   476.97 | 0.92%           |
| BAJAJ-AUTO.NS | TES             |         10046   | 10065.1  | 10107.1  | 0.61%           |
| ADANIENT.NS   | DES             |          2462   |  2464.95 |  2469.73 | 0.31%           |
| BAJFINANCE.NS | MA              |           958.6 |   977.07 |   960.89 | 0.24%           |
| AXISBANK.NS   

In [ ]:
# ============================================================
# BLOCK 8: ADAPTIVE MASTER FUSION
# FIXED:
#   - Uses _bulk_get cache — NO re-download
#   - Safe MultiIndex handling removed (cache returns flat DataFrames)
#   - All yfinance calls eliminated from this block
# ============================================================
import pandas as pd, numpy as np, os, json, warnings
from scipy.stats import norm, zscore
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
warnings.filterwarnings("ignore")

def run_adaptive_fusion():
    OUT_DIR = "/tmp/agentic_sector"
    try:
        df_quant    = pd.read_csv(os.path.join(OUT_DIR,"institutional_quant_signals.csv"))
        df_alerts   = pd.read_csv(os.path.join(OUT_DIR,"final_ml_alerts.csv"))
        df_articles = pd.read_csv(os.path.join(OUT_DIR,"clean_articles.csv"))
        df_5d       = pd.read_csv(os.path.join(OUT_DIR,"predictive_forecasts.csv"))
    except Exception as e:
        print(f"⚠️ Missing file: {e}. Run Blocks 1–7.1 first."); return

    tickers = df_quant["ticker"].tolist()
    model_col = "Winning Model" if "Winning Model" in df_5d.columns else "Winning_Model"
    winning_models = dict(zip(df_5d["ticker"], df_5d[model_col]))

    print(f"⏳ PHASE 1: 30-Day extrapolation using cached prices...")
    # Use cache from Block 7 — zero extra downloads
    all_data = _bulk_get(tickers, period="6mo")

    def fit_30d(prices, bm):
        cp = prices[-1]
        try:
            if bm == "MA":    return ARIMA(prices,order=(0,0,2)).fit().forecast(30)
            elif bm == "ARIMA": return ARIMA(prices,order=(1,1,1)).fit().forecast(30)
            elif bm == "SES": return ExponentialSmoothing(prices,trend=None,seasonal=None).fit().forecast(30)
            elif bm == "DES": return ExponentialSmoothing(prices,trend="add",seasonal=None).fit().forecast(30)
            elif bm == "TES":
                try: return ExponentialSmoothing(prices,trend="add",seasonal="add",seasonal_periods=5).fit().forecast(30)
                except: return ExponentialSmoothing(prices,trend="add",seasonal=None).fit().forecast(30)
        except: pass
        return [cp]*30

    base_forecasts = []
    for ticker in tickers:
        h = all_data.get(ticker, pd.DataFrame())
        if len(h) < 50: continue
        prices = h["Close"].values.astype(float); cp = prices[-1]
        bm = winning_models.get(ticker, "MA")
        fc = fit_30d(prices, bm)
        row = {
            "ticker": ticker,
            "Company": df_quant[df_quant["ticker"]==ticker]["company_name"].iloc[0] if ticker in df_quant["ticker"].values else ticker,
            "Current_Price": round(float(cp),2),
            "Winning_Model": bm,
            "Model_RMSE": float(df_5d[df_5d["ticker"]==ticker]["Model RMSE"].iloc[0]) if ticker in df_5d["ticker"].values else (float(df_5d["Model RMSE"].median()) if "Model RMSE" in df_5d.columns and not df_5d.empty else cp*0.015)
        }
        for i in range(1,31): row[f"Day_{i}"] = round(float(fc[i-1]),2)
        base_forecasts.append(row)

    df_30d = pd.DataFrame(base_forecasts)
    if df_30d.empty: print("⚠️ No 30D forecasts generated."); return

    print("🧬 PHASE 2: Adaptive Alpha Fusion...")
    ticker_map = df_articles[["title","ticker"]].drop_duplicates()
    merged     = df_alerts.merge(ticker_map, on="title", how="inner")
    sent_agg   = merged.groupby("ticker").agg(
        avg_z_score=("score","mean"), signal_count=("score","count")
    ).reset_index()

    master = df_30d.merge(sent_agg, on="ticker", how="left").fillna(0)
    master = master.merge(df_quant[["ticker","status","RVOL","quant_score","ATR"]], on="ticker", how="left")
    master.rename(columns={"status":"Quant_Status"}, inplace=True)
    master["Volatility_Pct"] = master["Model_RMSE"] / master["Current_Price"].clip(lower=1)
    master["ATR_Pct"] = master["ATR"].fillna(0) / master["Current_Price"].clip(lower=1)

    # Smooth confidence gate instead of a hard cutoff
    master["news_confident"] = 1.0 / (1.0 + np.exp(-(master["avg_z_score"].abs().fillna(0) - 0.5) * 4.0))
    master["Quant_Mag"] = master["quant_score"].abs()
    master["News_Mag"]  = (master["avg_z_score"].abs() * master["RVOL"]
                           * np.log1p(master["signal_count"] + 1) * master["news_confident"])

    zq = zscore(master["Quant_Mag"].fillna(0)); zn = zscore(master["News_Mag"].fillna(0))
    pq = norm.cdf(zq); pn = norm.cdf(zn)
    denom = pq + pn + 1e-9
    master["Math_Weight"] = pq / denom
    master["News_Weight"] = (pn / denom) * master["news_confident"]

    zqd = zscore(master["quant_score"].fillna(0)); zsd = zscore(master["avg_z_score"].fillna(0))
    master["Confluence_Index"]      = zqd * zsd
    master["Confluence_Multiplier"] = 1.0 + np.tanh(master["Confluence_Index"])

    base_score = (master["quant_score"]*master["Math_Weight"]) + (master["avg_z_score"]*master["News_Weight"])
    master["Raw_Master_Score"] = base_score * master["Confluence_Multiplier"]

    raw = master["Raw_Master_Score"].values
    raw_med = np.median(raw)
    raw_mad = np.median(np.abs(raw - raw_med))
    raw_scale = 1.4826 * raw_mad if raw_mad > 1e-9 else (np.std(raw) + 1e-9)
    master["Master_Score"] = (((raw - raw_med) / raw_scale) * 3.5).clip(-10,10).round(1)

    master["Vol_Expansion"]       = np.where(master["avg_z_score"].abs()>5.0, 1.0+((master["avg_z_score"].abs()-5.0)*0.20), 1.0)
    master["Effective_Volatility"]= master["Volatility_Pct"] * master["Vol_Expansion"]
    master["Daily_Alpha_Drift"]   = (master["avg_z_score"]/10.0) * master["Effective_Volatility"] * master["News_Weight"]
    master["Max_Limit"]           = master["Current_Price"] * 1.35
    master["Min_Limit"]           = master["Current_Price"] * 0.65

    decay_rate = np.clip(0.03 + master["ATR_Pct"] * 0.8, 0.03, 0.14)
    for t in range(1,31):
        decay = np.exp(-decay_rate*t)
        raw_alpha = master[f"Day_{t}"] * (1 + (master["Daily_Alpha_Drift"]*decay*t))
        master[f"Alpha_Day_{t}"] = np.clip(raw_alpha, master["Min_Limit"], master["Max_Limit"]).round(2)

    master["Base_Expected_Move"]  = ((master["Day_30"]  - master["Current_Price"])/master["Current_Price"]*100).round(2)
    master["Alpha_Expected_Move"] = ((master["Alpha_Day_30"] - master["Current_Price"])/master["Current_Price"]*100).round(2)

    ms_vals      = master["Master_Score"].values
    P_STRONG_BUY = float(np.percentile(ms_vals, 85))
    P_BUY        = float(np.percentile(ms_vals, 65))
    P_WATCH_BULL = float(np.percentile(ms_vals, 55))
    P_WATCH_BEAR = float(np.percentile(ms_vals, 45))
    P_SELL       = float(np.percentile(ms_vals, 35))
    P_STRONG_SELL= float(np.percentile(ms_vals, 15))

    print(f"   STRONG BUY≥{P_STRONG_BUY:.2f} | BUY≥{P_BUY:.2f} | SELL≤{P_SELL:.2f} | STRONG SELL≤{P_STRONG_SELL:.2f}")

    # Signal learning journal (proxy-based in batch mode)
    sigmoid = lambda x: 1.0 / (1.0 + np.exp(-x))
    learning = master[[
        "ticker","Company","Master_Score","Quant_Status","Base_Expected_Move","Alpha_Expected_Move",
        "Model_RMSE","Volatility_Pct","ATR_Pct"
    ]].copy()
    learning["Signal_Direction"] = np.where(learning["Master_Score"] >= 0, "BULLISH", "BEARISH")
    learning["Confidence_Proxy"] = sigmoid(np.abs(learning["Master_Score"]) / 3.0) * (1.0 - np.clip(learning["Volatility_Pct"], 0, 1))
    learning["Outcome_Proxy"] = np.sign(learning["Alpha_Expected_Move"].fillna(0)) * np.sign(learning["Base_Expected_Move"].fillna(0))
    learning["Learning_Weight"] = (
        sigmoid(np.abs(learning["Base_Expected_Move"].fillna(0)) / 4.0) *
        sigmoid(np.abs(learning["Alpha_Expected_Move"].fillna(0)) / 4.0) *
        np.clip(1.0 - learning["Volatility_Pct"].fillna(0), 0, 1)
    )
    learning.to_csv(os.path.join(OUT_DIR, "signal_learning.csv"), index=False)

    def make_signal(row):
        s  = row["Master_Score"]; q = str(row["Quant_Status"])
        am = row["Alpha_Expected_Move"]; bm = row["Base_Expected_Move"]
        if am > bm+2.0 and "BUY" in q:   return "🚀 HIGH PRIORITY BUY (News Boost)"
        if am < -1.0   and "BUY" in q:   return "🚨 HIGH PRIORITY RISK (Reversal)"
        if am > 1.0    and "SELL" in q:  return "👀 WATCHLIST (BULL) Reversal"
        if am < bm-2.0 and "SELL" in q:  return "🩸 STRONG SELL (News Shock)"
        if s >= P_STRONG_BUY: return "🚀 HIGH PRIORITY BUY"
        if s >= P_BUY:        return "📈 POTENTIAL BUY"
        if s >= P_WATCH_BULL: return "👀 WATCHLIST (BULL)"
        if s <= P_STRONG_SELL:return "🩸 STRONG SELL"
        if s <= P_SELL:       return "📉 POTENTIAL LOSS"
        if s <= P_WATCH_BEAR: return "⚠️ WATCHLIST (BEAR)"
        return "⚖️ NEUTRAL (Aligned)"

    master["Master_Signal"] = master.apply(make_signal, axis=1)

    thresh_meta = {
        "run_timestamp": pd.Timestamp.now().isoformat(),
        "P_STRONG_BUY":P_STRONG_BUY,"P_BUY":P_BUY,"P_WATCH_BULL":P_WATCH_BULL,
        "P_WATCH_BEAR":P_WATCH_BEAR,"P_SELL":P_SELL,"P_STRONG_SELL":P_STRONG_SELL,
        "n_stocks":len(master),"mean_score":round(float(np.mean(raw)),3),"std_score":round(float(np.std(raw)),3)
    }
    with open(os.path.join(OUT_DIR,"engine_meta.json"),"w") as f:
        json.dump(thresh_meta, f, indent=2)

    master = master.sort_values("Master_Score", ascending=False)
    master.to_csv(os.path.join(OUT_DIR,"master_crossover_signals.csv"), index=False)
    print("\n✅ Adaptive Master Fusion complete.")
    print(master["Master_Signal"].value_counts().to_markdown())
    print(master[["ticker","Master_Score","Master_Signal","Alpha_Expected_Move"]].head(10).to_markdown(index=False))

run_adaptive_fusion()


⏳ PHASE 1: 30-Day extrapolation using cached prices...
🧬 PHASE 2: Adaptive Alpha Fusion...
   STRONG BUY≥2.36 | BUY≥1.19 | SELL≤0.31 | STRONG SELL≤-3.09

✅ Adaptive Master Fusion complete.
| Master_Signal        |   count |
|:---------------------|--------:|
| 📈 POTENTIAL BUY     |      10 |
| 📉 POTENTIAL LOSS    |      10 |
| 🚀 HIGH PRIORITY BUY |       8 |
| 🩸 STRONG SELL       |       8 |
| 👀 WATCHLIST (BULL)  |       5 |
| ⚠️ WATCHLIST (BEAR)  |       5 |
| ⚖️ NEUTRAL (Aligned) |       4 |
| ticker        |   Master_Score | Master_Signal        |   Alpha_Expected_Move |
|:--------------|---------------:|:---------------------|----------------------:|
| BAJAJ-AUTO.NS |            6.7 | 🚀 HIGH PRIORITY BUY |                  7.95 |
| ONGC.NS       |            6.2 | 🚀 HIGH PRIORITY BUY |                  4.5  |
| BAJFINANCE.NS |            5.7 | 🚀 HIGH PRIORITY BUY |                  2    |
| ADANIENT.NS   |            5.5 | 🚀 HIGH PRIORITY BUY |                  2.01 |
| COALINDIA.N

In [ ]:
# ============================================================
# BLOCK 8.5: FRONTEND DATA WAREHOUSE
# FIXED:
#   - Uses _bulk_get cache — NO re-download
#   - pandas-ta: correct call signatures (ta.ema(h) not h.ta.ema())
#   - Chunked loop removed — data already in cache
# ============================================================
import pandas as pd, pandas_ta as ta, os, warnings
warnings.filterwarnings("ignore")

EXTENDED_HISTORY = False  # True = 1Y charts (slower)

def build_warehouse():
    print("🏗️ Building Frontend Data Warehouse (from cache)...")
    OUT_DIR = "/tmp/agentic_sector"
    try:
        df_master = pd.read_csv(os.path.join(OUT_DIR,"master_crossover_signals.csv"))
    except:
        print("⚠️ Run Block 8 first."); return

    tickers = df_master["ticker"].dropna().unique().tolist()
    period  = "1y" if EXTENDED_HISTORY else "6mo"

    if period == "6mo":
        print(f"📊 Using cached 6mo data for {len(tickers)} tickers (no download needed)")
        all_data = _bulk_get(tickers + ["^NSEI"], period="6mo")
    else:
        print(f"📡 Downloading 1Y extended data for {len(tickers)} tickers...")
        all_data = _yf_download(tickers + ["^NSEI"], period="1y")

    all_frames = []
    for ticker in tickers:
        h = all_data.get(ticker, pd.DataFrame())
        if h.empty: continue
        try:
            # Use functional API (avoids accessor deprecation)
            h = h.copy()
            ema20 = ta.ema(h["Close"], length=20)
            ema50 = ta.ema(h["Close"], length=50)
            sma20 = ta.sma(h["Close"], length=20)
            macd_df  = ta.macd(h["Close"])
            bb_df    = ta.bbands(h["Close"], length=20, std=2)
            rsi_s    = ta.rsi(h["Close"], length=14)
            atr_s    = ta.atr(h["High"], h["Low"], h["Close"], length=14)
            obv_s    = ta.obv(h["Close"], h["Volume"]) if "Volume" in h.columns else None

            if ema20 is not None: h["EMA_20"] = ema20
            if ema50 is not None: h["EMA_50"] = ema50
            if sma20 is not None: h["SMA_20"] = sma20
            if rsi_s  is not None: h["RSI"]   = rsi_s
            if atr_s  is not None: h["ATR"]   = atr_s
            if obv_s  is not None: h["OBV"]   = obv_s
            if macd_df is not None and not macd_df.empty:
                h["MACD"]       = macd_df.iloc[:,0]
                h["MACD_Signal"]= macd_df.iloc[:,1]
                h["MACD_Hist"]  = macd_df.iloc[:,2]
            if bb_df is not None and not bb_df.empty:
                h["BB_Upper"] = bb_df.iloc[:,0]
                h["BB_Mid"]   = bb_df.iloc[:,1]
                h["BB_Lower"] = bb_df.iloc[:,2]

            last10 = h["Close"].iloc[-10:].round(2).tolist()
            h["Last10Close"] = str(last10)
            h = h.reset_index(); h["ticker"] = ticker
            all_frames.append(h)
        except Exception as e:
            print(f"   ⚠️ {ticker}: {e}")

    if not all_frames:
        print("❌ No data collected."); return

    final = pd.concat(all_frames, ignore_index=True)
    final.to_csv(os.path.join(OUT_DIR,"historical_technicals.csv"), index=False)
    print(f"✅ Warehouse built: {len(final)} rows, {final['ticker'].nunique()} tickers")
    print(f"   Columns: {list(final.columns[:14])}")

build_warehouse()


🏗️ Building Frontend Data Warehouse (from cache)...
📊 Using cached 6mo data for 50 tickers (no download needed)
✅ Warehouse built: 6089 rows, 50 tickers
   Columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'EMA_20', 'EMA_50', 'SMA_20', 'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9', 'RSI_14', 'ATRr_14']


In [ ]:
# ============================================================
# ENGINE COMPLETE — Configuration Summary
# ============================================================
import json, os

OUT_DIR    = "/tmp/agentic_sector"
REPORT_DIR = os.path.join(OUT_DIR, "reports")
os.makedirs(REPORT_DIR, exist_ok=True)

config = {
    "data_dir": OUT_DIR,
    "files": {
        "master":    "master_crossover_signals.csv",
        "articles":  "clean_articles.csv",
        "alerts":    "final_ml_alerts.csv",
        "history":   "historical_technicals.csv",
        "quant":     "institutional_quant_signals.csv",
        "forecasts": "predictive_forecasts.csv",
        "meta":      "engine_meta.json",
        "reports":   "reports/",
    }
}
with open(os.path.join(OUT_DIR, "alphapulse_config.json"), "w") as f:
    json.dump(config, f, indent=2)

# Print summary of what was generated
print("\n✅ ENGINE COMPLETE — Output Summary")
print("=" * 50)
for name, fname in config["files"].items():
    fpath = os.path.join(OUT_DIR, fname)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        print(f"  ✓ {fname:<45} {size//1024:>4} KB")
    elif os.path.isdir(fpath):
        print(f"  ✓ {fname:<45} [dir]")
    else:
        print(f"  ✗ {fname:<45} [missing]")
print(f"\n📁 Output folder: {OUT_DIR}")
print("📊 Proceed to UI cells below.")



✅ ENGINE COMPLETE — Output Summary
  ✓ master_crossover_signals.csv                    39 KB
  ✓ clean_articles.csv                            1191 KB
  ✓ final_ml_alerts.csv                            790 KB
  ✓ historical_technicals.csv                     3400 KB
  ✓ institutional_quant_signals.csv                  5 KB
  ✓ predictive_forecasts.csv                         4 KB
  ✓ engine_meta.json                                 0 KB
  ✓ reports/                                      [dir]

📁 Output folder: /tmp/agentic_sector
📊 Proceed to UI cells below.



## Final launch section

This section writes the Streamlit app to `app.py` and starts it inside Colab.


In [ ]:

import os, sys, subprocess, time
from pathlib import Path

# Make sure the app can see the same outputs as the engine
os.environ["ATFINANCE_OUT_DIR"] = "/tmp/agentic_sector"
os.makedirs("/tmp/agentic_sector", exist_ok=True)
os.makedirs("/tmp/agentic_sector/reports", exist_ok=True)

print("ATfinance output directory ready:", os.environ["ATFINANCE_OUT_DIR"])


In [ ]:

# Write the Streamlit app file into the notebook runtime
from pathlib import Path

app_code = r'''
import os
from functools import lru_cache

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st

OUT_DIR = os.environ.get("ATFINANCE_OUT_DIR", "/tmp/agentic_sector")
REPORT_DIR = os.path.join(OUT_DIR, "reports")

st.set_page_config(page_title="ATfinance", page_icon="📈", layout="wide")

PALETTE = {
    "bg": "#13161B",
    "panel": "#1B2027",
    "panel_2": "#171B21",
    "text": "#F2EDE6",
    "muted": "#A8AFBD",
    "emerald": "#1FA87A",
    "royal": "#3F66F1",
    "maroon": "#7D3047",
    "gold": "#C7A46B",
    "line": "#2F3744",
    "soft": "#F7F2EA",
}

def safe_read_csv(path):
    try:
        if os.path.exists(path):
            return pd.read_csv(path)
    except Exception:
        pass
    return pd.DataFrame()

@lru_cache(maxsize=1)
def load_data():
    return {
        "quant": safe_read_csv(os.path.join(OUT_DIR, "institutional_quant_signals.csv")),
        "master": safe_read_csv(os.path.join(OUT_DIR, "master_crossover_signals.csv")),
        "alerts": safe_read_csv(os.path.join(OUT_DIR, "final_ml_alerts.csv")),
        "articles": safe_read_csv(os.path.join(OUT_DIR, "clean_articles.csv")),
        "history": safe_read_csv(os.path.join(OUT_DIR, "historical_technicals.csv")),
        "leaderboard": safe_read_csv(os.path.join(REPORT_DIR, "forecast_leaderboard.csv")),
        "forecasts": safe_read_csv(os.path.join(REPORT_DIR, "five_day_forecasts.csv")),
        "backtest": safe_read_csv(os.path.join(REPORT_DIR, "backtest_summary.csv")),
        "snapshot": safe_read_csv(os.path.join(REPORT_DIR, "explainability_snapshot.csv")),
        "learning": safe_read_csv(os.path.join(OUT_DIR, "signal_learning.csv")),
    }

def find_col(df, names):
    if df is None or df.empty:
        return None
    lower = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lower:
            return lower[n.lower()]
    for c in df.columns:
        if any(n.lower() in c.lower() for n in names):
            return c
    return None

def first(row, names, default=np.nan):
    if row is None:
        return default
    for n in names:
        if n in row.index and pd.notna(row[n]):
            return row[n]
    return default

def safe_num(x, default=np.nan):
    try:
        return float(x) if pd.notna(x) else default
    except Exception:
        return default

def money(x):
    try:
        return f"₹{float(x):,.0f}"
    except Exception:
        return "—"

def pct(x):
    try:
        return f"{float(x):+.2f}%"
    except Exception:
        return "—"

def header():
    st.markdown("""
    <div style='padding:18px 20px;border-radius:18px;background:linear-gradient(135deg,#171B21,#232A35);border:1px solid #2F3744'>
        <div style='font-size:32px;font-weight:800;color:#F7F2EA'>ATfinance</div>
        <div style='color:#A8AFBD;font-size:14px'>Luxury-style AI financial workspace — signals, quant, news, forecast, simulator, and learning.</div>
    </div>
    """, unsafe_allow_html=True)

def top_tickers():
    d = load_data()
    for key in ["master", "quant", "alerts", "history"]:
        df = d[key]
        if df.empty:
            continue
        c = find_col(df, ["ticker", "symbol"])
        if c:
            vals = df[c].astype(str).dropna().drop_duplicates().tolist()
            if vals:
                return vals
    return ["RELIANCE.NS", "TCS.NS", "HDFCBANK.NS"]

def get_ticker_history(ticker, horizon="6m"):
    df = load_data()["history"]
    if df.empty:
        return df
    tcol = find_col(df, ["ticker", "symbol"])
    if not tcol:
        return df.iloc[0:0]
    out = df[df[tcol].astype(str) == str(ticker)].copy()
    if out.empty:
        return out
    dcol = find_col(out, ["date", "datetime", "timestamp"])
    if dcol:
        out[dcol] = pd.to_datetime(out[dcol], errors="coerce")
        out = out.sort_values(dcol)
        days = {"1w":7, "2w":14, "3w":21, "1m":30, "3m":90, "6m":180}.get(horizon, 180)
        cut = pd.Timestamp.today().normalize() - pd.Timedelta(days=days)
        out = out[out[dcol] >= cut]
    return out.reset_index(drop=True)

def candle_chart(df, overlays):
    if df.empty:
        return None
    dcol = find_col(df, ["date", "datetime", "timestamp"])
    ocol = find_col(df, ["open"])
    hcol = find_col(df, ["high"])
    lcol = find_col(df, ["low"])
    ccol = find_col(df, ["close"])
    fig = go.Figure()
    x = df[dcol] if dcol else df.index
    if all([ocol, hcol, lcol, ccol]):
        fig.add_trace(go.Candlestick(x=x, open=df[ocol], high=df[hcol], low=df[lcol], close=df[ccol], name="Price"))
    elif ccol:
        fig.add_trace(go.Scatter(x=x, y=df[ccol], name="Price"))
    if ccol and "SMA20" in overlays:
        fig.add_trace(go.Scatter(x=x, y=df[ccol].rolling(20).mean(), name="SMA20"))
    if ccol and "SMA50" in overlays:
        fig.add_trace(go.Scatter(x=x, y=df[ccol].rolling(50).mean(), name="SMA50"))
    if ccol and "Bollinger" in overlays:
        mid = df[ccol].rolling(20).mean()
        std = df[ccol].rolling(20).std()
        fig.add_trace(go.Scatter(x=x, y=mid + 2*std, name="BB Upper"))
        fig.add_trace(go.Scatter(x=x, y=mid - 2*std, name="BB Lower"))
    fig.update_layout(template="plotly_dark", height=480, margin=dict(l=10,r=10,t=30,b=10), legend_orientation="h")
    return fig

def line_chart_from_cols(df, cols, title):
    fig = go.Figure()
    for c in cols:
        if c in df.columns:
            fig.add_trace(go.Scatter(y=df[c], mode="lines", name=c))
    fig.update_layout(template="plotly_dark", height=420, title=title, margin=dict(l=10,r=10,t=30,b=10))
    return fig

def sim_profit(base_price, amount, days, risk_mode, vol_pct):
    risk_mult = {"Conservative":0.75, "Balanced":1.0, "Aggressive":1.35}[risk_mode]
    drift_map = {"Conservative":0.0010, "Balanced":0.0018, "Aggressive":0.0030}
    mu = drift_map[risk_mode] * risk_mult
    sigma = max(0.01, min(0.15, vol_pct)) * risk_mult
    trials = 1000
    steps = np.random.normal(mu, sigma, size=(trials, days)).sum(axis=1)
    final_prices = base_price * np.exp(steps)
    profits = (final_prices - base_price) / base_price * amount
    return profits

st.markdown("""
<style>
.block-container { padding-top: 1rem; }
div[data-testid="stMetricValue"] { font-size: 1.6rem; }
</style>
""", unsafe_allow_html=True)

header()
d = load_data()

st.sidebar.title("ATfinance")
page = st.sidebar.radio("Navigate", ["Home","Master Crossover","Quant Analysis","News Intelligence","Forecast Lab","Simulator","Learning Hub"], index=0)
ticker = st.sidebar.selectbox("Ticker", top_tickers(), index=0)
horizon = st.sidebar.radio("Chart horizon", ["1w","2w","3w","1m","3m","6m"], horizontal=True, index=5)
overlays = st.sidebar.multiselect("Overlays", ["SMA20","SMA50","Bollinger","Base Forecast","Adjusted Forecast"], default=["SMA20","Bollinger"])
risk_mode = st.sidebar.selectbox("Risk mode", ["Conservative","Balanced","Aggressive"], index=1)
amount = st.sidebar.number_input("Investment amount", min_value=1000, max_value=10000000, value=10000, step=1000)
days = st.sidebar.slider("Holding days", 1, 60, 15)

master = d["master"]
quant = d["quant"]
alerts = d["alerts"]
articles = d["articles"]
history = get_ticker_history(ticker, horizon)
leader = d["leaderboard"]
forecasts = d["forecasts"]
learning = d["learning"]

if page == "Home":
    st.subheader("Market command center")
    if master.empty:
        st.info("Run the engine first.")
    else:
        tcol = find_col(master, ["ticker","symbol"])
        scol = find_col(master, ["Master_Score","master_score","score"])
        sigcol = find_col(master, ["Master_Signal","master_signal","signal"])
        pcol = find_col(master, ["Alpha_Expected_Move","expected_move","move"])
        top = master.sort_values(scol, ascending=False) if scol else master
        r = top.iloc[0]
        c1,c2,c3,c4 = st.columns(4)
        c1.metric("Top ticker", str(first(r,[tcol],"—")))
        c2.metric("Master score", f"{safe_num(first(r,[scol]),0):.2f}")
        c3.metric("Expected move", pct(first(r,[pcol])))
        c4.metric("Signal", str(first(r,[sigcol],"—")))
        st.write("Why bullish?", "Positive confluence across news, quant, and forecasts.")
        st.write("Why confidence high?", "Agreement between layers plus stable regime behavior.")
        topn = top.head(5)[[c for c in [tcol, scol, sigcol, pcol] if c]]
        st.dataframe(topn, use_container_width=True)

elif page == "Master Crossover":
    st.subheader("Master crossover")
    if master.empty:
        st.info("No master output.")
    else:
        tcol = find_col(master, ["ticker","symbol"])
        row = master[master[tcol].astype(str) == str(ticker)].head(1) if tcol else master.head(1)
        if not row.empty:
            r = row.iloc[0]
            c1,c2,c3,c4 = st.columns(4)
            c1.metric("Master score", f"{safe_num(first(r,['Master_Score','master_score','score']),0):.2f}")
            c2.metric("Direction", str(first(r,['Master_Signal','master_signal'],'—')))
            c3.metric("Alpha move", pct(first(r,['Alpha_Expected_Move','expected_move'])))
            c4.metric("Quant status", str(first(r,['Quant_Status','status'],'—')))
            st.write("Why bullish?", "Positive score, confluence, and better-than-base forecast.")
            st.write("Why forecast weak?", "Higher volatility or low cross-engine agreement can dilute confidence.")
            st.dataframe(row.T.rename(columns={row.columns[0]:"value"}), use_container_width=True)

elif page == "Quant Analysis":
    st.subheader("Quant studio")
    if quant.empty:
        st.info("No quant output.")
    else:
        tcol = find_col(quant, ["ticker","symbol"])
        qrow = quant[quant[tcol].astype(str) == str(ticker)] if tcol else quant.head(1)
        if not qrow.empty:
            q = qrow.iloc[0]
            c1,c2,c3,c4 = st.columns(4)
            c1.metric("Quant score", f"{safe_num(first(q,['quant_score']),0):.2f}")
            c2.metric("RSI", f"{safe_num(first(q,['RSI']),0):.2f}")
            c3.metric("ADX", f"{safe_num(first(q,['ADX']),0):.2f}")
            c4.metric("RVOL", f"{safe_num(first(q,['RVOL']),0):.2f}")
            st.caption("RSI measures momentum; MACD confirms trend; Bollinger shows volatility stretch.")
            if not history.empty:
                fig = candle_chart(history, overlays)
                if fig is not None:
                    st.plotly_chart(fig, use_container_width=True)
                if find_col(history, ["close", "Close"]):
                    st.line_chart(history[find_col(history, ["close", "Close"])])

elif page == "News Intelligence":
    st.subheader("News intelligence")
    if articles.empty:
        st.info("No articles loaded.")
    else:
        tcol = find_col(articles, ["ticker","symbol"])
        title_col = find_col(articles, ["title"])
        link_col = find_col(articles, ["link"])
        src_col = find_col(articles, ["source"])
        rel_col = find_col(articles, ["relevance_score"])
        pub_col = find_col(articles, ["published_at"])
        art = articles[articles[tcol].astype(str) == str(ticker)] if tcol else articles
        art = art.sort_values(rel_col, ascending=False) if rel_col in art.columns else art
        st.write(f"{len(art)} articles for {ticker}")
        for _, r in art.head(20).iterrows():
            st.markdown(
                f"- [{r.get(title_col,'Untitled')}]({r.get(link_col,'')})  \n"
                f"  <span style='color:#A8AFBD;font-size:12px'>{r.get(src_col,'')} | score {safe_num(r.get(rel_col,0),0):.2f} | {r.get(pub_col,'')}</span>",
                unsafe_allow_html=True,
            )

elif page == "Forecast Lab":
    st.subheader("Forecast lab")
    if forecasts.empty and leader.empty:
        st.info("No forecast outputs.")
    else:
        if not leader.empty:
            st.dataframe(leader.head(10), use_container_width=True)
        fdf = forecasts[forecasts[find_col(forecasts,["ticker","symbol"])].astype(str) == str(ticker)] if not forecasts.empty and find_col(forecasts,["ticker","symbol"]) else pd.DataFrame()
        if not fdf.empty:
            fig = go.Figure()
            day_cols = [c for c in fdf.columns if c.lower().startswith("day_")]
            if day_cols:
                base = [safe_num(fdf.iloc[0][c], np.nan) for c in day_cols if pd.notna(fdf.iloc[0][c])]
                fig.add_trace(go.Scatter(y=base, mode="lines+markers", name="Base Forecast"))
            adj_cols = [c for c in fdf.columns if "alpha_day" in c.lower()]
            if adj_cols:
                adj = [safe_num(fdf.iloc[0][c], np.nan) for c in adj_cols if pd.notna(fdf.iloc[0][c])]
                fig.add_trace(go.Scatter(y=adj, mode="lines+markers", name="Adjusted Forecast"))
            fig.update_layout(template="plotly_dark", height=420, margin=dict(l=10,r=10,t=30,b=10))
            st.plotly_chart(fig, use_container_width=True)

elif page == "Simulator":
    st.subheader("Scenario simulator")
    base_price = 100.0
    vol_pct = 0.03
    if not master.empty:
        tcol = find_col(master, ["ticker","symbol"])
        pcol = find_col(master, ["Current_Price","current_price","close"])
        mr = master[master[tcol].astype(str) == str(ticker)] if tcol else master.head(1)
        if not mr.empty and pcol:
            base_price = safe_num(mr.iloc[0].get(pcol), 100.0)
    if not quant.empty:
        qcol = find_col(quant, ["ticker","symbol"])
        qrow = quant[quant[qcol].astype(str) == str(ticker)] if qcol else quant.head(1)
        if not qrow.empty:
            rr = qrow.iloc[0]
            atr = safe_num(first(rr, ["ATR"]), np.nan)
            cp = safe_num(first(rr, ["Close"]), np.nan)
            if pd.notna(atr) and pd.notna(cp) and cp > 0:
                vol_pct = max(0.01, min(0.12, atr / cp))
    profits = sim_profit(base_price, amount, days, risk_mode, vol_pct)
    st.metric("Estimated profit (median)", money(np.median(profits)))
    c1,c2,c3,c4 = st.columns(4)
    c1.metric("Prob. profit", f"{(profits > 0).mean()*100:.1f}%")
    c2.metric("Prob. loss", f"{(profits <= 0).mean()*100:.1f}%")
    c3.metric("Worst case", money(np.percentile(profits, 5)))
    c4.metric("Best case", money(np.percentile(profits, 95)))
    st.write("Best exit window:", f"Day {max(1, days-3)} to Day {min(60, days+3)}")
    st.write("Confidence-adjusted return:", money(np.percentile(profits, 60) * 0.85))
    fig = go.Figure()
    fig.add_histogram(x=profits, nbinsx=40)
    fig.update_layout(template="plotly_dark", height=360, margin=dict(l=10,r=10,t=30,b=10))
    st.plotly_chart(fig, use_container_width=True)

elif page == "Learning Hub":
    st.subheader("Learning hub")
    st.write("Why bullish? Because multiple engines agree and the regime supports continuation.")
    st.write("Why confidence high? Because signal and forecast layers align.")
    st.write("Why forecast weak? Because volatility, sparse news, or low consensus widen the range.")
    st.write("RSI:", "Momentum oscillator. Above 70 can be stretched, below 30 can be weak.")
    st.write("MACD:", "Trend + momentum confirmation.")
    st.write("Bollinger:", "Volatility envelope around price.")
    st.write("RVOL:", "Relative volume above 1.8 often means unusually strong participation.")
    if not learning.empty:
        st.dataframe(learning.head(20), use_container_width=True)

st.caption("Cloudflare-ready: run this app via Streamlit and expose it with your preferred tunnel / domain setup.")
'''
Path("app.py").write_text(app_code, encoding="utf-8")
print("app.py written successfully.")


In [ ]:

# Install Streamlit only if it is missing
import importlib, subprocess, sys, time, os

try:
    import streamlit  # noqa: F401
    print("Streamlit already installed.")
except Exception:
    print("Installing Streamlit...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "streamlit"])

print("Ready to launch.")


In [ ]:

# Start Streamlit in the background and print a Colab proxy URL
import subprocess, sys, time, os, signal

# Kill any older streamlit process in this session if present
subprocess.run("pkill -f 'streamlit run app.py' || true", shell=True)

cmd = [
    sys.executable, "-m", "streamlit", "run", "app.py",
    "--server.headless", "true",
    "--server.address", "0.0.0.0",
    "--server.port", "8501",
]

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(8)

try:
    from google.colab import output as colab_output
    url = colab_output.eval_js("google.colab.kernel.proxyPort(8501)")
    print("\nATfinance UI is live here:")
    print(url)
except Exception:
    print("\nATfinance UI is running at http://localhost:8501")
    print("If this is Colab, the proxy URL should appear automatically once the runtime is ready.")



### Run order

1. Run the engine cells.
2. Run the final launch section.
3. Open the printed URL for the Streamlit UI.
